# 📊 Data Science Assignments — Complete Practice Workbook
### Retail Sales Dataset · 4,310 rows × 21 columns · 2020–2024

| Assignment | Topic | Marks |
|:---:|---|:---:|
| 1 | Data Cleaning | 100 |
| 2 | Exploratory Data Analysis | 100 |
| 3 | Statistical Analysis | 100 |
| 4 | Segmentation & Clustering | 100 |
| 5 | Time Series Forecasting | 100 |
| | **Total** | **500** |

> **Run cells top-to-bottom.** Assignment 1 produces `retail_sales_clean.csv` which every later assignment reads.
> Each section begins with a 📘 **Theory** block explaining the concept before the code.

## 🔧 Environment Setup

In [ ]:
# Run once to install any missing packages
import subprocess, sys
pkgs = ['pandas','numpy','matplotlib','seaborn','scipy','statsmodels',
        'scikit-learn','pmdarima','prophet']
for p in pkgs:
    subprocess.run([sys.executable,'-m','pip','install',p,'-q'], capture_output=True)
print("All packages ready.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi':110, 'figure.figsize':(12,5),
                     'axes.titlesize':13, 'axes.labelsize':11})

print("Imports complete ✓")

---
# 🧹 Assignment 1 — Data Cleaning
**Marks: 100 | Tasks: 8**

### 📘 Why Data Cleaning Matters
Real-world datasets are never clean. Before any analysis or modelling, you must:
- Understand the structure and completeness of the data
- Remove or fix invalid, duplicate, and inconsistent records
- Ensure every column has the correct data type

The `retail_sales_dataset.csv` was deliberately injected with:

| Issue | Columns Affected |
|---|---|
| ~80 exact duplicate rows | All columns |
| ~30 completely blank rows | All columns |
| Missing values (3–8%) | age, quantity, discount_pct, customer_satisfaction, days_to_ship |
| Mixed date formats (3 formats) | order_date |
| Inconsistent casing | gender, order_status |
| Outliers | age, quantity, days_to_ship |
| Wrong data types | return_flag (object instead of bool) |

---
## Task 1 — Initial Audit
📘 **Theory:** Before touching any data, run a *data audit* — a complete inventory of what you have. This gives you a baseline snapshot so you can measure exactly how much you cleaned.

Key audit steps:
1. **Shape** — how many rows and columns?
2. **dtypes** — are dates stored as strings? booleans as objects?
3. **Missing values** — which columns are incomplete and by how much?
4. **Duplicates** — are there exact copies of rows?

In [ ]:
# ── Load the raw dataset ──────────────────────────────────────────────────────
df_raw = pd.read_csv('retail_sales_dataset.csv')

print("=" * 55)
print(f"  Shape  : {df_raw.shape[0]:,} rows  ×  {df_raw.shape[1]} columns")
print("=" * 55)
print(f"\nColumn names:\n  {list(df_raw.columns)}")
print(f"\nData types:")
print(df_raw.dtypes.to_string())

In [ ]:
# ── Missing value audit ───────────────────────────────────────────────────────
missing_count = df_raw.isnull().sum()
missing_pct   = (missing_count / len(df_raw) * 100).round(2)

audit = pd.DataFrame({'Missing Count': missing_count,
                      'Missing %': missing_pct,
                      'Dtype': df_raw.dtypes}
                    ).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

print("Columns with missing values:")
print(audit.to_string())
print(f"\nTotal missing cells: {df_raw.isnull().sum().sum():,}")
print(f"Rows with at least 1 null: {df_raw.isnull().any(axis=1).sum():,}")

In [ ]:
# ── Missing value heatmap ─────────────────────────────────────────────────────
# Yellow stripe = missing cell; dark = present
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Heatmap
sns.heatmap(df_raw.isnull(), cbar=False, yticklabels=False,
            cmap='YlOrRd', ax=axes[0])
axes[0].set_title('Missing Values Heatmap\n(red stripe = missing cell)', fontsize=12)
axes[0].set_xlabel('Columns')

# Bar chart of missing %
cols_with_miss = missing_pct[missing_pct > 0].sort_values(ascending=True)
cols_with_miss.plot(kind='barh', color='steelblue', ax=axes[1])
axes[1].axvline(5, color='red', linestyle='--', label='5% threshold')
axes[1].set_xlabel('Missing %')
axes[1].set_title('Missing % per Column')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Duplicate detection ───────────────────────────────────────────────────────
n_exact_dupes = df_raw.duplicated().sum()
print(f"Exact duplicate rows: {n_exact_dupes}")

# Show 3 duplicate pairs
dup_mask = df_raw.duplicated(keep=False)
dupes = df_raw[dup_mask].sort_values(list(df_raw.columns[:4]))
print("\nSample duplicate rows (first 6 shown):")
print(dupes.head(6)[['order_id','order_date','customer_id','product_name','sales_amount']].to_string())

---
## Task 2 — Handle Duplicates
📘 **Theory:** Two types of duplicates require different treatment:

- **Exact duplicates** — every column identical. Safe to drop, keeping the first occurrence.
- **Near-duplicates** — same customer + date + product but different `order_id`. Could be a data entry error or a genuine second order. Your strategy choice must be *documented*.

`df.duplicated(keep='first')` marks all duplicate rows *except* the first occurrence as `True`.

In [ ]:
# ── Working copy ─────────────────────────────────────────────────────────────
df = df_raw.copy()
shape_before = df.shape

# Drop exact duplicates (keep first occurrence)
df = df.drop_duplicates(keep='first')
print(f"Rows before: {shape_before[0]:,}")
print(f"Rows after:  {df.shape[0]:,}")
print(f"Exact duplicates dropped: {shape_before[0] - df.shape[0]}")

In [ ]:
# ── Near-duplicate detection ──────────────────────────────────────────────────
# Near-dup = same customer_id + order_date + product_name but different order_id
near_dup_cols = ['customer_id', 'order_date', 'product_name']
near_dupes = df[df.duplicated(subset=near_dup_cols, keep=False)]
print(f"Near-duplicate groups (same customer+date+product): {len(near_dupes)}")

if len(near_dupes) > 0:
    print("\nSample near-duplicates:")
    print(near_dupes[near_dup_cols + ['order_id','sales_amount']].head(8).to_string())

# Strategy: keep the FIRST occurrence (earliest order_id sorts first)
# Rationale: duplicate entries are most likely data pipeline re-ingestion errors
df = df.drop_duplicates(subset=near_dup_cols, keep='first')
print(f"\nRows after near-dup removal: {df.shape[0]:,}")

---
## Task 3 — Handle Missing Values
📘 **Theory:** There is no universal rule for filling missing values — the right strategy depends on the column's *role* and *distribution*:

| Strategy | When to use |
|---|---|
| **Median** | Numeric column with outliers (outliers skew the mean) |
| **Mode** | Categorical or ordinal columns |
| **Constant (0)** | When absence of a value has clear business meaning |
| **Group-wise median** | When the distribution differs significantly across groups |
| **Drop the row** | When the column is a structural key that cannot be imputed |

> 💡 Always create a **flag column** (`col_missing = 1/0`) *before* imputing. This preserves information about missingness as a feature — it may be predictive in itself.

In [ ]:
# ── Drop fully blank rows (completely null rows injected into dataset) ─────────
n_all_null = df.isnull().all(axis=1).sum()
df = df.dropna(how='all')
print(f"Fully blank rows removed: {n_all_null}")

# ── Drop rows where structural keys are missing ────────────────────────────────
critical = ['order_date', 'order_status', 'region']
before = df.shape[0]
df = df.dropna(subset=critical)
print(f"Rows dropped (missing critical columns): {before - df.shape[0]}")
print(f"Remaining rows: {df.shape[0]:,}")

In [ ]:
# ── age: flag then fill with median ──────────────────────────────────────────
df['age_missing'] = df['age'].isnull().astype(int)

age_median = df['age'].median()
df['age'] = df['age'].fillna(age_median)

print(f"age: {df['age_missing'].sum()} rows were missing → filled with median ({age_median:.1f})")
print("Why median? The column has injected outliers (0, 150, 999) that distort the mean.")
print(f"  Mean before cleaning: {df_raw['age'].mean():.1f}  vs  Median: {age_median:.1f}")

In [ ]:
# ── customer_satisfaction: flag then fill with mode ──────────────────────────
df['satisfaction_missing'] = df['customer_satisfaction'].isnull().astype(int)

sat_mode = int(df['customer_satisfaction'].mode()[0])
df['customer_satisfaction'] = df['customer_satisfaction'].fillna(sat_mode)

print(f"customer_satisfaction: {df['satisfaction_missing'].sum()} missing → filled with mode ({sat_mode})")
print("Why mode? It is an ordinal 1-5 scale. Mode = most common rating.")

In [ ]:
# ── discount_pct: fill with 0 (business logic) ────────────────────────────────
n_disc_miss = df['discount_pct'].isnull().sum()
df['discount_pct'] = df['discount_pct'].fillna(0)
print(f"discount_pct: {n_disc_miss} missing → filled with 0")
print("Business logic: a missing discount record means no discount was applied.")

# ── quantity & days_to_ship: group-wise median by product_category ─────────────
for col in ['quantity', 'days_to_ship']:
    n_miss = df[col].isnull().sum()
    group_median = df.groupby('product_category')[col].transform('median')
    df[col] = df[col].fillna(group_median)
    # fallback: any remaining nulls (categories with all-null) use global median
    df[col] = df[col].fillna(df[col].median())
    print(f"{col}: {n_miss} missing → filled with group-wise median by product_category")

---
## Task 4 — Standardize Categorical Columns
📘 **Theory:** Categorical columns often accumulate *variant spellings* over time — different cases, abbreviations, or typos. These look like different categories to Python even though they represent the same thing.

`"Male"`, `"male"`, `"MALE"`, `"M"` → all mean the same thing but `df['gender'].value_counts()` would count them separately, inflating the number of categories and breaking any groupby analysis.

The fix: a **mapping dictionary** + `.replace()`, then verify with `.value_counts()` before and after.

In [ ]:
# ── gender: standardize all variants ─────────────────────────────────────────
print("BEFORE — gender value counts:")
print(df['gender'].value_counts().to_string())

gender_map = {
    'M':'Male', 'male':'Male', 'MALE':'Male', 'm':'Male',
    'F':'Female', 'female':'Female', 'FEMALE':'Female', 'f':'Female',
}
df['gender'] = df['gender'].replace(gender_map)

print("\nAFTER — gender value counts:")
print(df['gender'].value_counts().to_string())

In [ ]:
# ── order_status: strip whitespace, apply Title Case ─────────────────────────
print("BEFORE — order_status unique:", df['order_status'].unique())

df['order_status'] = df['order_status'].str.strip().str.title()

print("AFTER  — order_status unique:", df['order_status'].unique())

# ── region: validate only expected values exist ───────────────────────────────
valid_regions = {'North','South','East','West','Central'}
invalid_mask = ~df['region'].isin(valid_regions) & df['region'].notna()
print(f"\nInvalid region values found: {invalid_mask.sum()}")
print(df.loc[invalid_mask, 'region'].value_counts())
if invalid_mask.sum() > 0:
    df = df[~invalid_mask]
    print("Rows with invalid regions dropped.")

---
## Task 5 — Parse & Fix the Date Column
📘 **Theory:** `order_date` contains three different formats mixed together:
- `2021-05-12` → ISO format (`YYYY-MM-DD`)
- `12/05/2021` → Day-first format (`DD/MM/YYYY`)
- `"May 12, 2021"` → Long-form format

`pd.to_datetime()` with a single format will fail or silently produce wrong results on mixed formats. The robust solution: use `dateutil.parser.parse()` which handles all three automatically.

After parsing, extract **derived time features** — these will be essential for EDA and forecasting.

In [ ]:
from dateutil import parser as dateutil_parser

def safe_parse(val):
    """Parse a date string regardless of format. Returns NaT on failure."""
    if pd.isna(val):
        return pd.NaT
    try:
        return pd.Timestamp(dateutil_parser.parse(str(val), dayfirst=False))
    except Exception:
        return pd.NaT

# Parse — this may take ~5-10 seconds on 4k rows
df['order_date'] = df['order_date'].apply(safe_parse)

n_failed = df['order_date'].isna().sum()
print(f"Parsing complete. Failed / unparseable: {n_failed}")
print(f"dtype: {df['order_date'].dtype}")
print(f"Range: {df['order_date'].min().date()} to {df['order_date'].max().date()}")

In [ ]:
# ── Extract time features ─────────────────────────────────────────────────────
df['year']        = df['order_date'].dt.year
df['month']       = df['order_date'].dt.month
df['quarter']     = df['order_date'].dt.quarter
df['day_of_week'] = df['order_date'].dt.dayofweek      # 0=Mon, 6=Sun
df['is_weekend']  = df['day_of_week'].isin([5,6]).astype(int)

print("New time columns created:")
print(df[['order_date','year','month','quarter','day_of_week','is_weekend']].head(5).to_string())

# ── Validate date range ───────────────────────────────────────────────────────
out_of_range = df[(df['year'] < 2020) | (df['year'] > 2024)]
print(f"\nDates outside 2020-2024: {len(out_of_range)}")
df = df[(df['year'] >= 2020) & (df['year'] <= 2024)]
print(f"Rows after date validation: {df.shape[0]:,}")

---
## Task 6 — Outlier Detection & Treatment
📘 **Theory:** Outliers are values that deviate far from the bulk of observations. They can arise from:
- **Data entry errors** (e.g., age = 999)
- **System bugs** (e.g., quantity = −1)
- **Genuine extremes** (e.g., a bulk corporate order)

**Detection methods:**
| Method | Formula | Use when |
|---|---|---|
| **IQR** | Q1 − 1.5×IQR to Q3 + 1.5×IQR | Most numeric columns |
| **Domain knowledge** | e.g., age must be 18–90 | When valid range is known |
| **Z-score** | \|z\| > 3 | Approximately normal distributions |

**Treatment methods:**
- **Clip (cap)** — replace outlier with boundary value. Preserves row.
- **Remove** — delete the row. Use when value is operationally impossible.
- **Winsorize** — statistical clipping at percentile boundaries.

> ⚠️ **Do NOT remove negative `profit` values.** They represent loss-making orders — a valid business reality.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, col in zip(axes.flat, ['age','quantity','days_to_ship','sales_amount','profit','discount_pct']):
    df.boxplot(column=col, ax=ax, boxprops=dict(color='steelblue'),
               medianprops=dict(color='red',linewidth=2),
               flierprops=dict(marker='o',color='orange',markersize=4))
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('')

axes.flat[-1].set_visible(False) if len(axes.flat) > 5 else None
plt.suptitle('Boxplots Before Outlier Treatment — orange dots are outliers', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── age: clip to valid domain [18, 90] ───────────────────────────────────────
age_before = df['age'].copy()
df['age'] = df['age'].clip(lower=18, upper=90)
clipped = (age_before != df['age']).sum()
print(f"age: {clipped} values clipped to [18, 90]")
print(f"  Min before: {age_before.min():.0f}  →  After: {df['age'].min():.0f}")
print(f"  Max before: {age_before.max():.0f}  →  After: {df['age'].max():.0f}")

# ── quantity: remove impossible values ───────────────────────────────────────
invalid_qty = df[(df['quantity'] <= 0) | (df['quantity'] > 50)]
print(f"\nquantity: {len(invalid_qty)} invalid rows (≤0 or >50) removed")
df = df[(df['quantity'] > 0) & (df['quantity'] <= 50)]

# ── days_to_ship: replace impossibles with NaN then re-impute ─────────────────
invalid_ship = (df['days_to_ship'] < 1) | (df['days_to_ship'] > 30)
print(f"days_to_ship: {invalid_ship.sum()} values out of [1,30] → replaced with group median")
df.loc[invalid_ship, 'days_to_ship'] = np.nan
group_med = df.groupby('product_category')['days_to_ship'].transform('median')
df['days_to_ship'] = df['days_to_ship'].fillna(group_med).fillna(df['days_to_ship'].median())

In [ ]:
from scipy.stats.mstats import winsorize

# ── sales_amount: IQR outlier detection then Winsorize ───────────────────────
for col in ['sales_amount']:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f"{col}: {n_out} IQR outliers detected (bounds: {lo:,.0f} to {hi:,.0f})")
    df[col] = winsorize(df[col], limits=[0.01, 0.01])
    print(f"  Winsorized at 1st/99th percentile → new range: {df[col].min():,.0f} to {df[col].max():,.0f}")

# ── profit: show distribution (negatives are VALID — do not remove) ───────────
n_neg_profit = (df['profit'] < 0).sum()
print(f"\nprofit: {n_neg_profit} negative values — these are VALID loss-making orders.")
print("  We DO NOT remove them. They are important for profitability analysis.")
print(f"  profit range: {df['profit'].min():,.0f} to {df['profit'].max():,.0f}")

---
## Task 7 — Data Type Correction
📘 **Theory:** Pandas assigns default types on load — strings get `object`, everything else may become `float64` due to NaN coercion. Correct types:
- Reduce memory footprint (`Categorical` vs `object` can save 90%)
- Enable date-arithmetic on `datetime64`
- Enable boolean operations on `bool`
- Prevent silent numeric coercion in `Int64` (nullable integer preserves NaN without float)

In [ ]:
# ── order_date → datetime64 (already done in Task 5) ─────────────────────────
assert df['order_date'].dtype == 'datetime64[ns]', "order_date should be datetime64"
print(f"order_date dtype: {df['order_date'].dtype} ✓")

# ── customer_satisfaction → nullable Int64 ────────────────────────────────────
df['customer_satisfaction'] = df['customer_satisfaction'].astype('Int64')
print(f"customer_satisfaction dtype: {df['customer_satisfaction'].dtype} ✓")

# ── return_flag → bool ────────────────────────────────────────────────────────
# Handle string 'True'/'False' and actual bool
df['return_flag'] = df['return_flag'].map(
    {True:True, False:False, 'True':True, 'False':False, 1:True, 0:False}
).astype(bool)
print(f"return_flag dtype: {df['return_flag'].dtype} ✓")

# ── quantity, days_to_ship → int ──────────────────────────────────────────────
df['quantity']     = df['quantity'].round().astype(int)
df['days_to_ship'] = df['days_to_ship'].round().astype(int)
print(f"quantity dtype: {df['quantity'].dtype} ✓")
print(f"days_to_ship dtype: {df['days_to_ship'].dtype} ✓")

# ── High-cardinality strings → Categorical (memory optimization) ───────────────
mem_before = df.memory_usage(deep=True).sum() / 1024
for col in ['product_category','region','gender','payment_method','order_status']:
    df[col] = df[col].astype('category')
mem_after = df.memory_usage(deep=True).sum() / 1024

print(f"\nMemory before Categorical: {mem_before:,.1f} KB")
print(f"Memory after  Categorical: {mem_after:,.1f} KB")
print(f"Reduction: {(1 - mem_after/mem_before)*100:.1f}%")

---
## Task 8 — Final Validation & Export
📘 **Theory:** After cleaning, run a *data contract* check — a set of programmatic assertions that verify the cleaned data meets your quality standards. This is the same concept as unit tests in software engineering, applied to data.

If any check fails, you know exactly which column to revisit. A green validation report is your sign-off that the data is safe to pass downstream.

In [ ]:
def validate_dataframe(df):
    """Run data quality checks. Returns a pass/fail report DataFrame."""
    checks = []

    def chk(name, mask_fail, note=''):
        n = mask_fail.sum() if hasattr(mask_fail,'sum') else int(not mask_fail)
        checks.append({'Check': name, 'Failing Rows': n,
                       'Status': 'PASS ✅' if n == 0 else 'FAIL ❌', 'Note': note})

    # 1. No nulls in critical columns
    for col in ['order_id','order_date','region','product_category','sales_amount']:
        chk(f'No nulls in {col}', df[col].isnull())

    # 2. Age in valid range
    chk('age ∈ [18, 90]', (df['age'] < 18) | (df['age'] > 90))

    # 3. Quantity positive
    chk('quantity > 0', df['quantity'] <= 0)

    # 4. Gender valid
    chk('gender ∈ {Male,Female,Other}',
        ~df['gender'].isin(['Male','Female','Other']),
        'Unexpected gender values')

    # 5. Region valid
    chk('region ∈ 5 valid values',
        ~df['region'].isin(['North','South','East','West','Central']),
        'Unexpected region values')

    # 6. No future dates
    chk('No dates after 2024-12-31', df['order_date'] > pd.Timestamp('2024-12-31'))

    # 7. sales_amount positive
    chk('sales_amount > 0', df['sales_amount'] <= 0)

    return pd.DataFrame(checks)

report = validate_dataframe(df)
print(report.to_string(index=False))
print(f"\nOverall: {'ALL CHECKS PASSED ✅' if (report['Failing Rows']==0).all() else 'SOME CHECKS FAILED ❌'}")

In [ ]:
# ── Cleaning summary ─────────────────────────────────────────────────────────
print("=" * 55)
print("  DATA CLEANING SUMMARY")
print("=" * 55)
print(f"  Original shape  : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"  Cleaned shape   : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Rows removed    : {df_raw.shape[0] - df.shape[0]:,}")
print(f"  Columns added   : {df.shape[1] - df_raw.shape[1]} (flag + time features)")
print(f"  Missing cells   : {df.isnull().sum().sum()} (was {df_raw.isnull().sum().sum():,})")
print("=" * 55)

# ── Export clean dataset ──────────────────────────────────────────────────────
df.to_csv('retail_sales_clean.csv', index=False)
print("\n✅  Saved: retail_sales_clean.csv")

---
# 📊 Assignment 2 — Exploratory Data Analysis (EDA)
**Marks: 100 | Tasks: 6**

### 📘 What is EDA?
EDA is the process of *questioning the data visually and numerically* before applying any formal models. The goal is to:
- Understand the **distribution** of each variable
- Discover **relationships** between variables
- Identify **patterns**, **anomalies**, and **trends**
- Generate **hypotheses** that formal statistical tests will later confirm

> EDA is never just about making charts. Every chart must answer a specific question, and every answer must be translated into a business insight.

In [ ]:
# ── Load the cleaned dataset ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi':110, 'axes.titlesize':13})

df = pd.read_csv('retail_sales_clean.csv', parse_dates=['order_date'])

# Re-cast categoricals (lost on CSV round-trip)
for col in ['product_category','region','gender','payment_method','order_status']:
    df[col] = df[col].astype('category')

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(df.dtypes.to_string())

---
## Task 1 — Descriptive Statistics
📘 **Theory:**

- **Mean vs Median**: If mean >> median, the distribution is right-skewed (a few very large values pull the average up). Median is more representative of the "typical" value.
- **Skewness**: > 0 = right tail; < 0 = left tail. |skew| > 1 is considered highly skewed.
- **Kurtosis**: > 3 = heavy tails (more extreme values than a normal distribution). Excess kurtosis = kurtosis − 3.
- **Coefficient of Variation (CoV)**: std / mean — measures relative spread. Useful for comparing variability across columns with different units.

In [ ]:
numeric_cols = ['sales_amount','profit','age','discount_pct','days_to_ship',
                'quantity','unit_price','shipping_cost','customer_satisfaction']

# Full describe
print("=== df.describe() ===")
print(df[numeric_cols].describe().round(2).to_string())

In [ ]:
# ── Skewness, kurtosis, CoV summary table ─────────────────────────────────────
summary = pd.DataFrame({
    'Mean':     df[numeric_cols].mean(),
    'Median':   df[numeric_cols].median(),
    'Std':      df[numeric_cols].std(),
    'Min':      df[numeric_cols].min(),
    'Max':      df[numeric_cols].max(),
    'IQR':      df[numeric_cols].quantile(0.75) - df[numeric_cols].quantile(0.25),
    'Skewness': df[numeric_cols].skew(),
    'Kurtosis': df[numeric_cols].kurt(),
    'CoV %':    (df[numeric_cols].std() / df[numeric_cols].mean() * 100),
}).round(3)

print("=== Extended Summary Statistics ===")
print(summary.to_string())
print("\nMost right-skewed (high CoV):", summary['CoV %'].idxmax())
print("Highest kurtosis (heaviest tails):", summary['Kurtosis'].idxmax())

In [ ]:
# ── Visual: sorted skewness bar chart ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

summary['Skewness'].sort_values().plot(kind='barh', color=['#e74c3c' if v>0 else '#2ecc71'
    for v in summary['Skewness'].sort_values()], ax=axes[0])
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_title('Skewness per Column\n(red = right-skewed, green = left-skewed)')
axes[0].set_xlabel('Skewness')

summary['CoV %'].sort_values(ascending=False).plot(kind='barh', color='steelblue', ax=axes[1])
axes[1].set_title('Coefficient of Variation %\n(higher = more spread relative to mean)')
axes[1].set_xlabel('CoV %')

plt.tight_layout()
plt.show()

---
## Task 2 — Univariate Analysis
📘 **Theory:** Univariate analysis examines one variable at a time.

- **Histogram + KDE**: Shows frequency distribution. KDE (Kernel Density Estimate) is a smoothed version of the histogram.
- **Boxplot**: Shows median, IQR, and outliers. Box = IQR (25th–75th percentile); whiskers = 1.5×IQR; dots = outliers.
- **Bar chart**: For categorical variables, shows count per category.
- **Pie chart**: Shows proportion. Best for 2–5 categories.

In [ ]:
# ── Histograms with KDE ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
cols_hist = ['sales_amount','profit','age','discount_pct']
colors = ['#3498db','#2ecc71','#e67e22','#9b59b6']

for ax, col, color in zip(axes.flat, cols_hist, colors):
    sns.histplot(df[col].dropna(), kde=True, ax=ax, color=color, bins=40,
                 line_kws={'linewidth':2})
    ax.axvline(df[col].mean(),   color='red',   linestyle='--', label=f'Mean {df[col].mean():.0f}')
    ax.axvline(df[col].median(), color='black', linestyle=':',  label=f'Median {df[col].median():.0f}')
    ax.set_title(f'Distribution of {col}')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))

plt.suptitle('Histograms with KDE — Red dashed=Mean, Black dotted=Median', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()
print("\nObservation: sales_amount and profit are right-skewed (mean > median).")
print("This means a few very large orders are pulling the average up.")

In [ ]:
# ── Boxplots grid ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
box_cols = ['sales_amount','profit','age','discount_pct',
            'quantity','days_to_ship','unit_price','shipping_cost']

for ax, col in zip(axes.flat, box_cols):
    bp = df.boxplot(column=col, ax=ax,
                    boxprops=dict(color='steelblue'),
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='o', color='orange', markersize=3, alpha=0.5))
    q1, med, q3 = df[col].quantile([0.25,0.5,0.75])
    ax.set_title(f'{col}\nMedian={med:,.0f}  IQR={q3-q1:,.0f}', fontsize=9)
    ax.set_xlabel('')

plt.suptitle('Boxplots — Orange dots are outliers', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Bar charts for categorical columns ───────────────────────────────────────
cat_cols = ['product_category','region','payment_method','gender','order_status']
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

palette = sns.color_palette('Set2', 10)
for ax, col in zip(axes.flat, cat_cols):
    counts = df[col].value_counts()
    counts.sort_values().plot(kind='barh', ax=ax, color=palette[:len(counts)])
    ax.set_title(f'{col} — Order Count')
    ax.set_xlabel('Count')
    for i, v in enumerate(counts.sort_values()):
        ax.text(v + 5, i, f'{v:,}', va='center', fontsize=9)

axes.flat[-1].set_visible(False)
plt.suptitle('Categorical Column Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Pie chart: return rate ────────────────────────────────────────────────────
ret_counts = df['return_flag'].value_counts()
labels = ['Not Returned', 'Returned']
colors = ['#2ecc71', '#e74c3c']
explode = [0, 0.08]

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(ret_counts, labels=labels, colors=colors,
                                   autopct='%1.1f%%', explode=explode,
                                   startangle=90, textprops={'fontsize':12})
autotexts[0].set_fontsize(14)
autotexts[1].set_fontsize(14)
ax.set_title(f'Return Rate\nTotal Orders: {len(df):,}', fontsize=13)
plt.tight_layout()
plt.show()
print(f"Return rate: {df['return_flag'].mean()*100:.1f}%")

---
## Task 3 — Bivariate Analysis
📘 **Theory:** Bivariate analysis examines the relationship between **two variables** at a time.

| Chart type | Best for |
|---|---|
| Scatter plot | Continuous vs continuous — shows direction, strength, linearity |
| Box plot (grouped) | Continuous vs categorical — compares distributions |
| Bar chart (grouped) | Categorical vs categorical — compares counts |
| Heatmap | Two categoricals vs a numeric aggregate |

Always ask: "Is this relationship **meaningful** for a business decision?" 

In [ ]:
# ── Scatter: sales_amount vs profit, coloured by product_category ─────────────
fig, ax = plt.subplots(figsize=(12, 6))
categories = df['product_category'].cat.categories
palette = sns.color_palette('tab10', len(categories))

for cat, color in zip(categories, palette):
    mask = df['product_category'] == cat
    ax.scatter(df.loc[mask,'sales_amount'], df.loc[mask,'profit'],
               alpha=0.35, s=18, color=color, label=cat)

ax.set_xlabel('Sales Amount (INR)')
ax.set_ylabel('Profit (INR)')
ax.set_title('Sales Amount vs Profit — coloured by Product Category')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

corr = df['sales_amount'].corr(df['profit'])
print(f"Pearson r (sales vs profit): {corr:.3f}  — {'strong positive' if corr>0.7 else 'moderate'} relationship")

In [ ]:
# ── Box: sales_amount by region ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
region_order = df.groupby('region')['sales_amount'].median().sort_values(ascending=False).index

df.boxplot(column='sales_amount', by='region', ax=ax,
           order=region_order,
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='red', linewidth=2.5),
           flierprops=dict(marker='.', alpha=0.3))

# Annotate medians
for i, region in enumerate(region_order):
    med = df.loc[df['region']==region, 'sales_amount'].median()
    ax.text(i+1, med*1.03, f'₹{med:,.0f}', ha='center', fontsize=9, color='red')

ax.set_title('Sales Amount Distribution by Region')
ax.set_xlabel('Region'); ax.set_ylabel('Sales Amount (INR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
plt.suptitle('')
plt.tight_layout()
plt.show()

print("Median sales by region:")
print(df.groupby('region')['sales_amount'].median().sort_values(ascending=False).apply(lambda x: f'₹{x:,.0f}'))

In [ ]:
# ── Bar: avg profit margin by product_category ────────────────────────────────
df['profit_margin_pct'] = df['profit'] / df['sales_amount'] * 100

margin_by_cat = df.groupby('product_category')['profit_margin_pct'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in margin_by_cat]
bars = margin_by_cat.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Average Profit Margin % by Product Category')
ax.set_xlabel('Category'); ax.set_ylabel('Avg Profit Margin %')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
for i, v in enumerate(margin_by_cat):
    ax.text(i, v + 0.5 if v >= 0 else v - 2, f'{v:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print(f"Most profitable category:  {margin_by_cat.idxmax()} ({margin_by_cat.max():.1f}%)")
print(f"Least profitable category: {margin_by_cat.idxmin()} ({margin_by_cat.min():.1f}%)")

In [ ]:
# ── Scatter: discount_pct vs sales_amount ────────────────────────────────────
from scipy import stats as sp_stats

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(df['discount_pct'], df['sales_amount'], alpha=0.2, s=15, color='#9b59b6')

# Regression line
slope, intercept, r, p, _ = sp_stats.linregress(df['discount_pct'], df['sales_amount'])
x_line = np.linspace(0, 0.4, 100)
ax.plot(x_line, slope*x_line + intercept, color='red', linewidth=2,
        label=f'r = {r:.3f}  (p = {p:.4f})')
ax.set_xlabel('Discount %'); ax.set_ylabel('Sales Amount (INR)')
ax.set_title('Discount % vs Sales Amount — Does offering a bigger discount drive higher sales?')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
ax.legend()
plt.tight_layout()
plt.show()
print(f"r = {r:.3f} → {'positive' if r>0 else 'negative'} relationship")
print("Interpretation: " + ("Higher discounts correlate with higher sales value." if r>0
      else "Higher discounts do NOT drive higher sales — interesting!"))

---
## Task 4 — Time Series & Trend Analysis
📘 **Theory:** Time series analysis looks at how a variable changes *over time*.

Key patterns to look for:
- **Trend**: Long-term increase or decrease
- **Seasonality**: Regular, repeating patterns (e.g., Q4 holiday boost)
- **Irregularity**: One-off spikes or dips (e.g., sales event, supply chain disruption)

**Resampling**: Converting transaction-level data into a regular time series by summing/averaging over time buckets (daily → monthly).

In [ ]:
# ── Set order_date as index and resample monthly ──────────────────────────────
df_ts = df.set_index('order_date').sort_index()
monthly = df_ts['sales_amount'].resample('ME').sum()
rolling_3m = monthly.rolling(window=3, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly.index, monthly.values, color='steelblue', alpha=0.6,
        linewidth=1.5, label='Monthly Sales')
ax.plot(rolling_3m.index, rolling_3m.values, color='red', linewidth=2.5,
        label='3-Month Rolling Mean')
ax.fill_between(monthly.index, monthly.values, alpha=0.1, color='steelblue')
ax.set_title('Monthly Sales Amount — 2020 to 2024', fontsize=13)
ax.set_xlabel('Month'); ax.set_ylabel('Total Sales (INR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))
ax.legend()
plt.tight_layout()
plt.show()
print(f"Total sales: ₹{monthly.sum()/1e6:.1f}M  |  Avg monthly: ₹{monthly.mean()/1e6:.2f}M")
print(f"Peak month: {monthly.idxmax().strftime('%b %Y')} (₹{monthly.max()/1e6:.2f}M)")

In [ ]:
# ── Multi-year overlay: compare seasonality across years ─────────────────────
monthly_df = monthly.reset_index()
monthly_df.columns = ['date','sales']
monthly_df['year']  = monthly_df['date'].dt.year
monthly_df['month'] = monthly_df['date'].dt.month

fig, ax = plt.subplots(figsize=(13, 5))
colors_yr = {2020:'#e74c3c',2021:'#f39c12',2022:'#2ecc71',2023:'#3498db',2024:'#9b59b6'}
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

for yr, grp in monthly_df.groupby('year'):
    ax.plot(grp['month'], grp['sales'], marker='o', linewidth=2,
            color=colors_yr[yr], label=str(yr), markersize=5)

ax.set_xticks(range(1,13))
ax.set_xticklabels(month_names)
ax.set_title('Monthly Sales by Year — Seasonal Pattern Comparison')
ax.set_ylabel('Total Sales (INR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))
ax.legend(title='Year')
plt.tight_layout()
plt.show()
print("Observation: Check if Q4 (Oct-Dec) consistently shows a peak across all years.")

In [ ]:
# ── Month-over-Month growth rate ──────────────────────────────────────────────
mom_growth = monthly.pct_change() * 100

fig, ax = plt.subplots(figsize=(14, 5))
colors_bar = ['#2ecc71' if v >= 0 else '#e74c3c' for v in mom_growth.dropna()]
ax.bar(mom_growth.dropna().index, mom_growth.dropna().values,
       color=colors_bar, width=20, alpha=0.8)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Month-over-Month (MoM) Sales Growth %')
ax.set_ylabel('Growth %')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}%'))
plt.tight_layout()
plt.show()

print("Top 3 MoM growth months:")
print(mom_growth.nlargest(3).apply(lambda x: f'{x:.1f}%'))
print("\nTop 5 highest-revenue months:")
print(monthly.nlargest(5).apply(lambda x: f'₹{x/1e6:.2f}M'))

---
## Task 5 — Multivariate Analysis
📘 **Theory:** Multivariate analysis examines **three or more variables simultaneously**.

- **Correlation matrix**: Measures pairwise linear relationships. Values: −1 (perfect negative) to +1 (perfect positive). 0 = no linear relationship.
- **Pair plot**: A grid of scatter plots for every pair of variables + histograms on the diagonal.
- **Pivot table heatmap**: Cross-tabulate two categorical variables with a numeric aggregate — great for spotting which category-region combinations perform best.

In [ ]:
# ── Correlation matrix heatmap ────────────────────────────────────────────────
num_cols = ['sales_amount','profit','age','discount_pct','quantity',
            'unit_price','days_to_ship','customer_satisfaction','shipping_cost']
corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # show lower triangle
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, square=True, ax=ax,
            linewidths=0.5, mask=mask)
ax.set_title('Pearson Correlation Matrix\n(only lower triangle shown)', fontsize=12)
plt.tight_layout()
plt.show()

# Print top correlations
corr_unstacked = corr_matrix.where(np.tril(np.ones(corr_matrix.shape), k=-1).astype(bool))
top = corr_unstacked.abs().unstack().dropna().sort_values(ascending=False).head(5)
print("Top 5 strongest pairwise correlations:")
for (c1,c2), v in top.items():
    print(f"  {c1} ↔ {c2}: {corr_matrix.loc[c1,c2]:.3f}")

In [ ]:
# ── Pivot heatmap: avg sales by region × product_category ────────────────────
pivot = df.pivot_table(values='sales_amount',
                       index='region',
                       columns='product_category',
                       aggfunc='mean')

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(pivot, annot=True, fmt=',.0f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label':'Avg Sales (INR)'})
ax.set_title('Average Sales Amount (INR) — Region × Product Category')
ax.set_xlabel('Product Category'); ax.set_ylabel('Region')
plt.tight_layout()
plt.show()

best_region, best_cat = np.unravel_index(pivot.values.argmax(), pivot.shape)
print(f"Highest avg sales: Region={pivot.index[best_region]}, "
      f"Category={pivot.columns[best_cat]} (₹{pivot.values.max():,.0f})")

---
## Task 6 — Business Insights Report
📘 **Theory:** EDA must produce *actionable intelligence*, not just charts.

For each insight below, we follow the pattern:
1. **Question** — what are we trying to know?
2. **Evidence** — chart + number
3. **Insight** — what does the data tell us?
4. **Action** — what should the business *do* with this information?

In [ ]:
# ── Insight 1: Revenue & Profit by product_category ──────────────────────────
cat_summary = df.groupby('product_category').agg(
    total_revenue=('sales_amount','sum'),
    total_profit=('profit','sum'),
    order_count=('order_id','count')
).sort_values('total_revenue', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cat_summary['total_revenue'].div(1e6).sort_values().plot(
    kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Total Revenue by Category (₹M)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x:.0f}M'))

cat_summary['total_profit'].div(1e6).sort_values().plot(
    kind='barh', ax=axes[1], color='#2ecc71')
axes[1].set_title('Total Profit by Category (₹M)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x:.0f}M'))

plt.tight_layout()
plt.show()
print(f"Highest Revenue: {cat_summary['total_revenue'].idxmax()}")
print(f"Highest Profit:  {cat_summary['total_profit'].idxmax()}")

In [ ]:
# ── Insight 2: Return rate by region ─────────────────────────────────────────
return_by_region = df.groupby('region').agg(
    total_orders=('order_id','count'),
    returned=('return_flag','sum')
)
return_by_region['return_rate_pct'] = (return_by_region['returned']
                                        / return_by_region['total_orders'] * 100).round(2)
return_by_region = return_by_region.sort_values('return_rate_pct', ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(return_by_region.index, return_by_region['return_rate_pct'],
              color=['#e74c3c' if v == return_by_region['return_rate_pct'].max()
                     else 'steelblue' for v in return_by_region['return_rate_pct']])
ax.axhline(return_by_region['return_rate_pct'].mean(), color='orange',
           linestyle='--', label=f"Avg {return_by_region['return_rate_pct'].mean():.1f}%")
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f'{bar.get_height():.1f}%', ha='center', fontsize=10)
ax.set_title('Return Rate % by Region'); ax.set_ylabel('Return Rate %')
ax.legend()
plt.tight_layout()
plt.show()
print(return_by_region[['total_orders','returned','return_rate_pct']].to_string())

In [ ]:
# ── Insight 3: Top 10 cities by revenue ──────────────────────────────────────
city_revenue = df.groupby('city')['sales_amount'].sum().nlargest(10).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
city_revenue.plot(kind='barh', color='darkcyan', ax=ax)
for i, (city, val) in enumerate(city_revenue.items()):
    ax.text(val * 1.01, i, f'₹{val/1e6:.2f}M', va='center', fontsize=9)
ax.set_title('Top 10 Cities by Total Sales Revenue')
ax.set_xlabel('Total Sales (INR)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.0f}M'))
plt.tight_layout()
plt.show()
print(f"Top city: {city_revenue.idxmax()} with ₹{city_revenue.max()/1e6:.2f}M in sales")

In [ ]:
# ── Insight 4: Payment method vs avg order value ─────────────────────────────
pay_stats = df.groupby('payment_method')['sales_amount'].agg(['mean','sem']).round(2)
pay_stats.columns = ['mean','sem']
pay_stats = pay_stats.sort_values('mean', ascending=False)
ci95 = pay_stats['sem'] * 1.96  # 95% CI half-width

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(pay_stats.index, pay_stats['mean'], yerr=ci95, capsize=5,
       color='darkorchid', alpha=0.8, ecolor='black')
ax.set_title('Average Order Value by Payment Method\n(error bars = 95% CI)')
ax.set_ylabel('Avg Sales Amount (INR)')
ax.set_xticklabels(pay_stats.index, rotation=20, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x:,.0f}'))
plt.tight_layout()
plt.show()
print(f"Highest avg order value: {pay_stats['mean'].idxmax()} (₹{pay_stats['mean'].max():,.0f})")

---
# 📐 Assignment 3 — Statistical Analysis
**Marks: 100 | Tasks: 7**

### 📘 Why Formal Statistics?
EDA tells you *what* patterns look like. Statistics tells you *whether those patterns are real* or could be explained by random chance.

**Key concepts:**
- **Null hypothesis (H₀)**: The "boring" default — no effect, no difference.
- **p-value**: Probability of seeing your result *or more extreme* if H₀ is true. p < 0.05 → reject H₀.
- **Effect size**: *How large* is the difference (practical significance vs statistical significance).
- **Type I error (α)**: Rejecting a true H₀ (false positive). We set α = 0.05.
- **Type II error (β)**: Failing to reject a false H₀ (false negative).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi':110})

df = pd.read_csv('retail_sales_clean.csv', parse_dates=['order_date'])
for col in ['product_category','region','gender','payment_method','order_status']:
    df[col] = df[col].astype('category')

print(f"Loaded {df.shape[0]:,} rows. Ready for statistical analysis.")

---
## Task 1 — Distribution Testing (Normality)
📘 **Theory:** Many statistical tests (t-test, ANOVA, OLS regression) assume the data or residuals are *normally distributed*. We must verify this before applying parametric tests.

**Three normality tests:**

| Test | H₀ | Suitable for |
|---|---|---|
| **Shapiro-Wilk** | Data is normally distributed | n < 5,000 (very sensitive) |
| **D'Agostino-Pearson** | Skewness = 0 and kurtosis = 3 | Any sample size |
| **Q-Q Plot** | Points fall on a straight diagonal line | Visual check |

**Log transformation**: If a column is right-skewed, `log(x+1)` often brings it closer to normal by compressing the long right tail.

In [ ]:
def normality_report(series, name, sample_n=500):
    """Run three normality tests and print a structured report."""
    # Use a random sample for Shapiro-Wilk (designed for small-medium n)
    s = series.dropna()
    sample = s.sample(min(sample_n, len(s)), random_state=42)

    sw_stat,  sw_p  = sp.shapiro(sample)
    dp_stat,  dp_p  = sp.normaltest(sample)
    print(f"\n{'='*50}")
    print(f"  Normality Test: {name}")
    print(f"  n = {len(s):,}  |  sample for Shapiro = {len(sample)}")
    print(f"{'='*50}")
    print(f"  Shapiro-Wilk   : W={sw_stat:.4f}  p={sw_p:.6f}  "
          f"{'→ NOT Normal ❌' if sw_p<0.05 else '→ Normal ✅'}")
    print(f"  D'Agostino-P.  : stat={dp_stat:.4f}  p={dp_p:.6f}  "
          f"{'→ NOT Normal ❌' if dp_p<0.05 else '→ Normal ✅'}")
    print(f"  Skewness: {s.skew():.3f}  Kurtosis: {s.kurt():.3f}")

for col in ['sales_amount','profit','age']:
    normality_report(df[col], col)

In [ ]:
# ── Q-Q plots for 3 columns ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, col in enumerate(['sales_amount','profit','age']):
    # Original
    sp.probplot(df[col].dropna(), dist='norm', plot=axes[0, i])
    axes[0, i].set_title(f'Q-Q Plot: {col}\n(original)')

    # Log-transformed
    log_data = np.log1p(df[col].dropna().clip(lower=0))
    sp.probplot(log_data, dist='norm', plot=axes[1, i])
    axes[1, i].set_title(f'Q-Q Plot: log(1+{col})')

plt.suptitle('Q-Q Plots — Points on diagonal line = Normal distribution', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()
print("Interpretation: If points deviate from the red line (especially at tails),")
print("the distribution is non-normal. Log-transformation often improves this.")

In [ ]:
# ── Log transformation of sales_amount ────────────────────────────────────────
df['log_sales'] = np.log1p(df['sales_amount'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df['sales_amount'], kde=True, bins=50, ax=axes[0], color='steelblue')
axes[0].set_title(f'sales_amount  (skew={df["sales_amount"].skew():.2f})')

sns.histplot(df['log_sales'], kde=True, bins=50, ax=axes[1], color='darkorange')
axes[1].set_title(f'log(1+sales_amount)  (skew={df["log_sales"].skew():.2f})')

plt.suptitle('Log Transformation Effect on sales_amount', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()
print("After log-transform, skewness reduced from",
      f"{df['sales_amount'].skew():.2f} to {df['log_sales'].skew():.2f}")

---
## Task 2 — Confidence Intervals
📘 **Theory:** A **95% confidence interval (CI)** means: if we repeated this sampling process 100 times, approximately 95 of the resulting intervals would contain the true population mean.

**Formula (t-interval for mean):**
$$CI = \bar{x} \pm t_{\alpha/2,\,n-1} \cdot \frac{s}{\sqrt{n}}$$

Where `t` is the critical value from the t-distribution (≈ 1.96 for large n), `s` is the sample std, and `n` is sample size.

**Non-overlapping CIs** between two groups suggest their means are *statistically significantly different*.

In [ ]:
# ── Overall 95% CI for mean sales_amount ─────────────────────────────────────
def mean_ci_95(series):
    s = series.dropna()
    lo, hi = sp.t.interval(0.95, df=len(s)-1,
                            loc=s.mean(), scale=sp.sem(s))
    return s.mean(), lo, hi

overall_mean, lo, hi = mean_ci_95(df['sales_amount'])
print(f"Overall mean sales_amount:  ₹{overall_mean:,.2f}")
print(f"95% Confidence Interval:   [₹{lo:,.2f}  to  ₹{hi:,.2f}]")
print(f"CI width: ₹{hi-lo:,.2f}")
print("\nInterpretation: We are 95% confident the true mean sales_amount")
print(f"in the population lies between ₹{lo:,.0f} and ₹{hi:,.0f}.")

In [ ]:
# ── 95% CI per region — error bar chart ──────────────────────────────────────
region_ci = {}
for region in df['region'].cat.categories:
    sales = df.loc[df['region']==region, 'sales_amount']
    mean, lo, hi = mean_ci_95(sales)
    region_ci[region] = {'mean': mean, 'lo': lo, 'hi': hi,
                         'err_lo': mean-lo, 'err_hi': hi-mean}

ci_df = pd.DataFrame(region_ci).T.sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(ci_df.index, ci_df['mean'],
            yerr=[ci_df['err_lo'], ci_df['err_hi']],
            fmt='o', capsize=8, capthick=2, elinewidth=2,
            markersize=10, color='steelblue', ecolor='#e74c3c')

for i, (region, row) in enumerate(ci_df.iterrows()):
    ax.annotate(f"₹{row['mean']:,.0f}", (i, row['mean']),
                textcoords='offset points', xytext=(12,0), fontsize=9)

ax.set_title('Mean Sales Amount by Region\n(Error bars = 95% Confidence Interval)')
ax.set_ylabel('Mean Sales Amount (INR)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'₹{x:,.0f}'))
plt.tight_layout()
plt.show()

print("\nDo any CIs NOT overlap? Non-overlapping CIs → likely statistically significant difference.")
print("Overlapping CIs → cannot conclude a significant difference without a formal test.")

---
## Task 3 — Two-Sample Hypothesis Tests
📘 **Theory — Choosing the right test:**

| Test | When to use |
|---|---|
| **Independent t-test** | Comparing 2 group means; both groups approx. normal, large n |
| **Welch's t-test** | Like t-test but does NOT assume equal variances (safer default) |
| **Mann-Whitney U** | Non-parametric alternative; use when data is not normal or ordinal |
| **Levene's test** | First check: are the two group variances equal? |

**Template for every test:**
1. State H₀ and H₁
2. Check normality (Shapiro-Wilk)
3. If normal → t-test; if not → Mann-Whitney U
4. Report: test statistic, p-value, conclusion at α=0.05
5. Compute effect size (Cohen's d)

In [ ]:
def two_sample_report(group1, group2, label1, label2, var_name):
    """Full two-sample test report with automatic test selection."""
    print(f"\n{'='*60}")
    print(f"  {var_name}: {label1} vs {label2}")
    print(f"  n₁={len(group1):,}  mean₁=₹{group1.mean():,.0f}")
    print(f"  n₂={len(group2):,}  mean₂=₹{group2.mean():,.0f}")
    print(f"{'='*60}")

    # Normality
    _, sw_p1 = sp.shapiro(group1.sample(min(200, len(group1)), random_state=1))
    _, sw_p2 = sp.shapiro(group2.sample(min(200, len(group2)), random_state=1))
    normal = (sw_p1 > 0.05) and (sw_p2 > 0.05)

    # Levene
    _, lev_p = sp.levene(group1, group2)
    equal_var = lev_p > 0.05
    print(f"  Shapiro-Wilk: {label1} p={sw_p1:.4f}, {label2} p={sw_p2:.4f}")
    print(f"  Levene's (equal variance): p={lev_p:.4f}  → {'Equal' if equal_var else 'Unequal'} variance")

    # Choose test
    if normal:
        stat, p = sp.ttest_ind(group1, group2, equal_var=equal_var)
        test_name = "Welch's t-test" if not equal_var else "Independent t-test"
    else:
        stat, p = sp.mannwhitneyu(group1, group2, alternative='two-sided')
        test_name = "Mann-Whitney U"

    # Effect size (Cohen's d)
    pooled_std = np.sqrt((group1.std()**2 + group2.std()**2) / 2)
    cohens_d = (group1.mean() - group2.mean()) / pooled_std
    effect = 'small' if abs(cohens_d)<0.5 else ('medium' if abs(cohens_d)<0.8 else 'large')

    print(f"  Test chosen: {test_name}")
    print(f"  H₀: mean({label1}) = mean({label2})")
    print(f"  H₁: mean({label1}) ≠ mean({label2})")
    print(f"  Statistic={stat:.4f}  p-value={p:.6f}")
    print(f"  Decision: {'REJECT H₀ ✅' if p<0.05 else 'FAIL TO REJECT H₀'} (α=0.05)")
    print(f"  Cohen's d = {cohens_d:.3f} → {effect} effect")

# Test A: Male vs Female sales_amount
g1 = df.loc[df['gender']=='Male',  'sales_amount']
g2 = df.loc[df['gender']=='Female','sales_amount']
two_sample_report(g1, g2, 'Male','Female','sales_amount')

In [ ]:
# Test B: returned vs not-returned customer_satisfaction (Mann-Whitney: ordinal)
g1 = df.loc[df['return_flag']==True,  'customer_satisfaction'].dropna()
g2 = df.loc[df['return_flag']==False, 'customer_satisfaction'].dropna()
stat, p = sp.mannwhitneyu(g1, g2, alternative='two-sided')
print(f"\ncustomer_satisfaction: Returned vs Not Returned")
print(f"H₀: satisfaction is the same for returned and non-returned orders")
print(f"Mann-Whitney U = {stat:.2f}  p = {p:.6f}")
print(f"Decision: {'REJECT H₀ ✅ — significant difference' if p<0.05 else 'No significant difference'}")
print(f"Mean satisfaction — Returned: {g1.mean():.2f}  Not Returned: {g2.mean():.2f}")

# Test C: Electronics vs Clothing profit
g1 = df.loc[df['product_category']=='Electronics','profit']
g2 = df.loc[df['product_category']=='Clothing',   'profit']
stat, p = sp.ttest_ind(g1, g2, equal_var=False)
print(f"\nprofit: Electronics vs Clothing  (Welch's t-test)")
print(f"H₀: mean profit is equal for Electronics and Clothing")
print(f"t = {stat:.4f}  p = {p:.6f}")
print(f"Decision: {'REJECT H₀ ✅' if p<0.05 else 'Fail to Reject H₀'}")
print(f"Mean profit — Electronics: ₹{g1.mean():,.0f}  Clothing: ₹{g2.mean():,.0f}")

---
## Task 4 — ANOVA (Analysis of Variance)
📘 **Theory:** ANOVA tests whether **3 or more group means are equal** simultaneously.

**Why not multiple t-tests?** If you run 5 t-tests at α=0.05, the probability of a false positive (Type I error) inflates to ~23%. ANOVA tests all groups at once, keeping α=0.05.

**F-statistic** = Variance *between* groups ÷ Variance *within* groups. Large F → groups differ more than we'd expect by chance.

**Post-hoc (Tukey HSD)**: ANOVA only tells you *some* groups differ. Tukey's Honestly Significant Difference test tells you *which* pairs differ, with multiple-comparison correction.

In [ ]:
# ── Check ANOVA assumptions first ────────────────────────────────────────────
print("=== ANOVA Assumption Check: sales_amount across regions ===")
print("\n1. Normality (Shapiro-Wilk per group):")
region_groups = [df.loc[df['region']==r, 'sales_amount'].sample(min(200,sum(df['region']==r)),
                 random_state=42) for r in df['region'].cat.categories]

for region, grp in zip(df['region'].cat.categories, region_groups):
    _, p = sp.shapiro(grp)
    print(f"   {region}: p={p:.4f}  {'Normal ✅' if p>0.05 else 'NOT Normal ❌'}")

print("\n2. Homogeneity of Variance (Levene's test):")
lev_stat, lev_p = sp.levene(*region_groups)
print(f"   Levene's: F={lev_stat:.4f}  p={lev_p:.4f}  "
      f"{'Equal variances ✅' if lev_p>0.05 else 'Unequal variances ❌'}")
print("   Note: ANOVA is robust to mild violations with large equal samples.")

In [ ]:
# ── One-Way ANOVA: sales_amount across 5 regions ─────────────────────────────
groups_by_region = [df.loc[df['region']==r,'sales_amount'] for r in df['region'].cat.categories]
f_stat, p_val = sp.f_oneway(*groups_by_region)

print(f"One-Way ANOVA: sales_amount ~ region")
print(f"H₀: μ_North = μ_South = μ_East = μ_West = μ_Central")
print(f"H₁: At least one region mean differs")
print(f"F = {f_stat:.4f}  p = {p_val:.6f}")
print(f"Decision: {'REJECT H₀ ✅ — at least one region mean significantly differs' if p_val<0.05 else 'Fail to Reject H₀'}")

# Eta-squared (effect size)
grand_mean = df['sales_amount'].mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups_by_region)
ss_total   = sum((df['sales_amount'] - grand_mean)**2)
eta_sq = ss_between / ss_total
print(f"\nEta-squared (η²) = {eta_sq:.4f}")
print(f"Interpretation: {eta_sq*100:.1f}% of total variance in sales_amount is explained by region.")

In [ ]:
# ── Tukey HSD post-hoc test ───────────────────────────────────────────────────
tukey = pairwise_tukeyhsd(df['sales_amount'], df['region'], alpha=0.05)
print(tukey.summary())

# Visualise Tukey results
fig, ax = plt.subplots(figsize=(8, 6))
tukey.plot_simultaneous(ax=ax, comparison_name=df['region'].cat.categories[0])
ax.set_title('Tukey HSD 95% Confidence Intervals\nNon-overlapping intervals = significant difference')
plt.tight_layout()
plt.show()

In [ ]:
# ── Two-Way ANOVA: sales_amount ~ region * product_category ──────────────────
model = smf.ols('sales_amount ~ C(region) + C(product_category) + C(region):C(product_category)',
                data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print("Two-Way ANOVA Table:")
print(anova_table.round(4).to_string())
print("\nInterpretation:")
for idx, row in anova_table.iterrows():
    print(f"  {idx[:40]:<40} p = {row['PR(>F)']:.4f}  "
          f"{'Significant ✅' if row['PR(>F)']<0.05 else 'Not significant'}")

---
## Task 5 — Correlation Analysis
📘 **Theory — Three correlation methods:**

| Method | Formula type | Use when |
|---|---|---|
| **Pearson r** | Linear | Both variables continuous, roughly normal |
| **Spearman ρ** | Rank-based | Monotonic relationship, ordinal, or non-normal |
| **Point-Biserial** | Pearson with 0/1 variable | One binary, one continuous |

**Effect size for r:**
- Small: |r| < 0.1
- Medium: 0.1 ≤ |r| < 0.3
- Large: |r| ≥ 0.5

In [ ]:
# ── Pearson: sales_amount vs profit ──────────────────────────────────────────
r, p = sp.pearsonr(df['sales_amount'], df['profit'])
print(f"Pearson r (sales_amount vs profit): {r:.4f}  p = {p:.2e}")
effect = 'small' if abs(r)<0.1 else ('medium' if abs(r)<0.3 else ('large' if abs(r)<0.5 else 'very large'))
print(f"Effect size: {effect}  |  {r*100:.1f}% of variance shared between the two variables (r²)")

# ── Spearman: customer_satisfaction vs discount_pct ──────────────────────────
rho, p = sp.spearmanr(df['customer_satisfaction'].dropna(),
                       df.loc[df['customer_satisfaction'].notna(),'discount_pct'])
print(f"\nSpearman ρ (satisfaction vs discount_pct): {rho:.4f}  p = {p:.4f}")
print("Why Spearman? satisfaction is ordinal (1-5 scale) — rank correlation is more appropriate.")
print(f"Direction: {'negative' if rho<0 else 'positive'} — {'higher discount → lower satisfaction' if rho<0 else 'higher discount → higher satisfaction'}")

# ── Point-Biserial: return_flag vs sales_amount ───────────────────────────────
pb_r, p = sp.pointbiserialr(df['return_flag'].astype(int), df['sales_amount'])
print(f"\nPoint-Biserial r (return_flag vs sales_amount): {pb_r:.4f}  p = {p:.4f}")
print(f"Direction: {'Returned orders have HIGHER sales amounts' if pb_r>0 else 'Returned orders have LOWER sales amounts'}")

---
## Task 6 — Linear Regression
📘 **Theory:** Linear regression models the relationship between a dependent variable (y = `sales_amount`) and one or more independent variables (x).

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \ldots + \varepsilon$$

**Key outputs from OLS summary:**
- **R²**: Proportion of variance in y explained by the model (0 = none, 1 = perfect)
- **p-values**: Is each coefficient significantly different from 0?
- **VIF** (Variance Inflation Factor): Multicollinearity check. VIF > 10 = problematic.

**4 Regression Assumptions** (must verify after fitting):
1. Linearity — residuals vs fitted plot should show no pattern
2. Normality of residuals — Q-Q plot should be diagonal
3. Homoscedasticity — residuals should have equal spread at all fitted values
4. Independence — Durbin-Watson statistic should be near 2.0

In [ ]:
# ── Simple OLS: sales_amount ~ unit_price ────────────────────────────────────
model1 = smf.ols('sales_amount ~ unit_price', data=df).fit()
print(model1.summary())

In [ ]:
# ── Multiple OLS: add quantity, discount_pct, days_to_ship ───────────────────
model2 = smf.ols('sales_amount ~ unit_price + quantity + discount_pct + days_to_ship',
                 data=df).fit()
print(f"R² = {model2.rsquared:.4f}  Adj R² = {model2.rsquared_adj:.4f}")
print(f"F-statistic = {model2.fvalue:.4f}  p = {model2.f_pvalue:.4e}")
print("\nCoefficients:")
print(model2.params.round(4).to_string())

# VIF check
X = df[['unit_price','quantity','discount_pct','days_to_ship']].dropna()
vif_data = pd.DataFrame({
    'Feature': X.columns,
    'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
print("\nVariance Inflation Factors (VIF > 10 = multicollinearity problem):")
print(vif_data.to_string(index=False))

In [ ]:
# ── Regression assumption diagnostics ────────────────────────────────────────
fitted  = model2.fittedvalues
resid   = model2.resid
std_res = (resid - resid.mean()) / resid.std()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. Residuals vs Fitted
axes[0,0].scatter(fitted, resid, alpha=0.3, s=12, color='steelblue')
axes[0,0].axhline(0, color='red', linewidth=1.5)
axes[0,0].set_xlabel('Fitted Values'); axes[0,0].set_ylabel('Residuals')
axes[0,0].set_title('Residuals vs Fitted\n(no pattern = linearity holds)')

# 2. Q-Q plot of residuals
sp.probplot(resid, dist='norm', plot=axes[0,1])
axes[0,1].set_title('Normal Q-Q of Residuals\n(diagonal = normal residuals)')

# 3. Scale-Location (√|std_residuals| vs fitted)
axes[1,0].scatter(fitted, np.sqrt(np.abs(std_res)), alpha=0.3, s=12, color='green')
axes[1,0].set_xlabel('Fitted Values'); axes[1,0].set_ylabel('√|Std Residuals|')
axes[1,0].set_title('Scale-Location\n(flat line = homoscedasticity)')

# 4. Residuals histogram
sns.histplot(resid, kde=True, bins=40, ax=axes[1,1], color='darkorange')
axes[1,1].set_title('Residual Distribution\n(bell shape = normal)')

plt.suptitle('OLS Regression Diagnostic Plots', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Durbin-Watson
from statsmodels.stats.stattools import durbin_watson
dw = durbin_watson(resid)
print(f"Durbin-Watson statistic: {dw:.3f}  (target: ~2.0 for independence)")
print(f"Interpretation: {'OK ✅' if 1.5 < dw < 2.5 else 'Possible autocorrelation ⚠️'}")

---
## Task 7 — Chi-Square Tests
📘 **Theory:** Chi-square tests check whether two **categorical** variables are *independent* of each other.

**H₀**: Variable A is independent of Variable B.
**H₁**: They are NOT independent — knowing one tells you something about the other.

**Cramér's V** = effect size:
$$V = \sqrt{\frac{\chi^2}{n \cdot \min(r-1, c-1)}}$$
- V < 0.1: weak association
- 0.1 ≤ V < 0.3: moderate
- V ≥ 0.3: strong

In [ ]:
def chi2_report(var1, var2, data):
    """Full chi-square independence test with Cramér's V."""
    ct = pd.crosstab(data[var1], data[var2])
    chi2, p, dof, expected = sp.chi2_contingency(ct)

    n = ct.sum().sum()
    min_dim = min(ct.shape[0]-1, ct.shape[1]-1)
    cramers_v = np.sqrt(chi2 / (n * min_dim))
    effect = 'weak' if cramers_v<0.1 else ('moderate' if cramers_v<0.3 else 'strong')

    print(f"\n{'='*55}")
    print(f"  Chi-Square: {var1} vs {var2}")
    print(f"{'='*55}")
    print(f"  H₀: {var1} is independent of {var2}")
    print(f"  χ² = {chi2:.4f}  df = {dof}  p = {p:.6f}")
    print(f"  Decision: {'REJECT H₀ ✅ — significant association' if p<0.05 else 'Fail to Reject H₀'}")
    print(f"  Cramér's V = {cramers_v:.4f} → {effect} association")
    return ct

ct1 = chi2_report('return_flag',    'product_category', df)
ct2 = chi2_report('payment_method', 'region',           df)
ct3 = chi2_report('gender',         'product_category', df)

In [ ]:
# ── Visualise: return rate by product_category ────────────────────────────────
ret_by_cat = df.groupby('product_category')['return_flag'].mean() * 100
ret_by_cat = ret_by_cat.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c' if v > ret_by_cat.mean() else 'steelblue' for v in ret_by_cat]
ret_by_cat.plot(kind='bar', ax=ax, color=colors)
ax.axhline(ret_by_cat.mean(), color='black', linestyle='--',
           label=f'Avg {ret_by_cat.mean():.1f}%')
ax.set_title('Return Rate % by Product Category\n(red = above average)')
ax.set_ylabel('Return Rate %')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.legend()
for i, v in enumerate(ret_by_cat):
    ax.text(i, v+0.2, f'{v:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---
# 🔵 Assignment 4 — Segmentation & Clustering
**Marks: 100 | Tasks: 6**

### 📘 Why Segment Customers?
Not all customers are equal. A business cannot treat a customer who spent ₹5L last week the same as one who last purchased 2 years ago and spent ₹500 total.

Segmentation partitions customers into groups that share similar behaviours, enabling:
- **Targeted marketing** (different message per segment)
- **Resource allocation** (focus retention budget on high-value at-risk customers)
- **Product personalisation**

We use two approaches:
1. **RFM** — rule-based scoring (interpretable, business-driven)
2. **Unsupervised ML** — K-Means, Hierarchical, DBSCAN (data-driven, finds hidden patterns)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi':110})

df = pd.read_csv('retail_sales_clean.csv', parse_dates=['order_date'])
for col in ['product_category','region','gender','payment_method','order_status']:
    df[col] = df[col].astype('category')
print(f"Loaded {df.shape[0]:,} rows")

---
## Task 1 — RFM Analysis
📘 **Theory:** RFM is the most widely used customer segmentation framework in retail.

| Metric | Definition | Business meaning |
|---|---|---|
| **Recency (R)** | Days since last purchase | How recently did they shop? |
| **Frequency (F)** | Number of orders | How often do they shop? |
| **Monetary (M)** | Total spend | How much have they spent? |

**Scoring (1–5):** Each metric is divided into 5 equal-frequency bins (quintiles). Score 5 = best behaviour.
- R score: **lower recency days = score 5** (purchased recently)
- F score: **higher frequency = score 5**
- M score: **higher spend = score 5**

In [ ]:
# ── Compute R, F, M per customer ─────────────────────────────────────────────
REF_DATE = pd.Timestamp('2025-01-01')

rfm = df.groupby('customer_id').agg(
    recency   = ('order_date',   lambda x: (REF_DATE - x.max()).days),
    frequency = ('order_id',     'count'),
    monetary  = ('sales_amount', 'sum')
).reset_index()

print("RFM Table (first 10 rows):")
print(rfm.head(10).to_string(index=False))
print(f"\nShape: {rfm.shape}")
print(f"\nRFM Summary:")
print(rfm[['recency','frequency','monetary']].describe().round(1).to_string())

In [ ]:
# ── Score each metric 1-5 using quintiles ────────────────────────────────────
# Recency: LOWER days = BETTER = higher score → reverse labels
rfm['R'] = pd.qcut(rfm['recency'],   q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'].rank(method='first'),  q=5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_Score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)

print("RFM with scores (first 10 rows):")
print(rfm[['customer_id','recency','frequency','monetary','R','F','M','RFM_Score']].head(10).to_string(index=False))
print(f"\nUnique RFM scores: {rfm['RFM_Score'].nunique()}")

In [ ]:
# ── Map scores to named segments ─────────────────────────────────────────────
def rfm_segment(score):
    if score in ['555','554','545','544','535']:
        return 'Champions'
    elif score in ['543','444','435','445','543','453']:
        return 'Loyal Customers'
    elif score in ['512','513','414','415','512','511']:
        return 'Potential Loyalists'
    elif score in ['311','411','331']:
        return 'Promising'
    elif score in ['255','254','245','244','253','252']:
        return 'At Risk'
    elif score in ['155','154','145','144']:
        return 'Cannot Lose'
    elif score in ['111','112','121','131','141']:
        return 'Hibernating'
    else:
        return 'Others'

rfm['segment'] = rfm['RFM_Score'].apply(rfm_segment)

seg_summary = rfm.groupby('segment').agg(
    count=('customer_id','count'),
    avg_recency=('recency','mean'),
    avg_frequency=('frequency','mean'),
    avg_monetary=('monetary','mean'),
    total_value=('monetary','sum')
).round(1).sort_values('total_value', ascending=False)

print("Segment Summary:")
print(seg_summary.to_string())

In [ ]:
# ── Visualise segment distribution ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: count
seg_counts = rfm['segment'].value_counts()
colors_seg = sns.color_palette('Set2', len(seg_counts))
seg_counts.plot(kind='barh', ax=axes[0], color=colors_seg)
axes[0].set_title('Customer Count per Segment')
axes[0].set_xlabel('Number of Customers')

# Bubble chart: Recency vs Frequency, size = Monetary
seg_agg = rfm.groupby('segment').agg(
    avg_recency=('recency','mean'),
    avg_frequency=('frequency','mean'),
    total_monetary=('monetary','sum')
).reset_index()

scatter = axes[1].scatter(seg_agg['avg_recency'], seg_agg['avg_frequency'],
                           s=seg_agg['total_monetary']/5000,
                           c=range(len(seg_agg)), cmap='tab10', alpha=0.7)
for _, row in seg_agg.iterrows():
    axes[1].annotate(row['segment'], (row['avg_recency'], row['avg_frequency']),
                     fontsize=8, ha='center')
axes[1].set_xlabel('Avg Recency (days — lower is better)')
axes[1].set_ylabel('Avg Frequency (orders)')
axes[1].set_title('RFM Bubble Chart\n(bubble size = total monetary value)')

plt.tight_layout()
plt.show()

---
## Task 2 — Feature Engineering for ML Clustering
📘 **Theory:** Machine learning clustering algorithms work on a **feature matrix** — one row per customer, multiple numeric columns.

We need to:
1. **Aggregate** transaction-level data to customer-level
2. **Encode** categorical features (preferred category, payment method)
3. **Scale** all numeric features to the same range

**Why scaling?** K-Means uses Euclidean distance. If `total_sales` is in thousands and `avg_satisfaction` is 1–5, the distance will be dominated entirely by `total_sales`, effectively ignoring satisfaction. StandardScaler makes every feature have mean=0 and std=1.

In [ ]:
# ── Build customer-level feature table ───────────────────────────────────────
def safe_mode(s):
    m = s.mode()
    return m.iloc[0] if len(m) > 0 else np.nan

cust_features = df.groupby('customer_id').agg(
    total_orders    = ('order_id',             'count'),
    total_sales     = ('sales_amount',          'sum'),
    total_profit    = ('profit',                'sum'),
    avg_discount    = ('discount_pct',          'mean'),
    avg_satisfaction= ('customer_satisfaction', 'mean'),
    return_rate     = ('return_flag',           'mean'),
    avg_days_ship   = ('days_to_ship',          'mean'),
    avg_age         = ('age',                   'mean'),
    pref_category   = ('product_category', safe_mode),
    pref_payment    = ('payment_method',   safe_mode),
).reset_index()

print(f"Customer feature table: {cust_features.shape}")
print(cust_features.head(5).to_string(index=False))

In [ ]:
# ── One-hot encode categoricals + scale numerics ─────────────────────────────
# One-hot encode preferred category and payment
cust_enc = pd.get_dummies(cust_features,
                           columns=['pref_category','pref_payment'],
                           drop_first=True)

# Fill any remaining NaN
numeric_feat_cols = [c for c in cust_enc.columns if c != 'customer_id']
cust_enc[numeric_feat_cols] = cust_enc[numeric_feat_cols].fillna(
    cust_enc[numeric_feat_cols].median())

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cust_enc[numeric_feat_cols])

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Features: {numeric_feat_cols[:10]} ... ({len(numeric_feat_cols)} total)")
print(f"\nAfter scaling — mean per feature (should be ~0):")
print(pd.Series(X_scaled.mean(axis=0), index=numeric_feat_cols).round(3).head(8).to_string())

---
## Task 3 — K-Means Clustering
📘 **Theory:** K-Means assigns each point to its nearest cluster centroid, then recomputes centroids, iterating until convergence.

**Choosing k:**
- **Elbow Method**: Plot inertia (WCSS — within-cluster sum of squares) vs k. The "elbow" is where adding more clusters gives diminishing returns.
- **Silhouette Score**: Measures how similar a point is to its own cluster vs other clusters. Range: −1 to +1. Higher = better separation.

**Limitation:** K-Means assumes spherical clusters of similar size. It's sensitive to outliers and requires k to be pre-specified.

In [ ]:
# ── Elbow Method ─────────────────────────────────────────────────────────────
inertias, sil_scores = [], []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels, sample_size=1000, random_state=42))
    print(f"  k={k}  inertia={km.inertia_:,.0f}  silhouette={sil_scores[-1]:.4f}")

In [ ]:
# ── Plot Elbow and Silhouette ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Elbow
axes[0].plot(K_range, inertias, 'bo-', markersize=8, linewidth=2)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method\nLook for the "elbow" — diminishing returns')
axes[0].set_xticks(list(K_range))

# Silhouette
best_k = list(K_range)[sil_scores.index(max(sil_scores))]
axes[1].plot(K_range, sil_scores, 'rs-', markersize=8, linewidth=2)
axes[1].axvline(best_k, color='green', linestyle='--', label=f'Best k={best_k}')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis\nHigher is better (max=1.0)')
axes[1].set_xticks(list(K_range))
axes[1].legend()

plt.suptitle('Choosing Optimal k for K-Means', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()
print(f"\nOptimal k by silhouette: {best_k} (score={max(sil_scores):.4f})")

In [ ]:
# ── Fit final K-Means ─────────────────────────────────────────────────────────
OPTIMAL_K = best_k
km_final = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(X_scaled)
cust_features['cluster'] = cluster_labels

print(f"K-Means with k={OPTIMAL_K}")
print(f"Cluster sizes:")
print(cust_features['cluster'].value_counts().sort_index().to_string())

# ── PCA to 2D for visualisation ───────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_

print(f"\nPCA: 2 components explain {explained.sum()*100:.1f}% of variance")
print(f"  PC1: {explained[0]*100:.1f}%  PC2: {explained[1]*100:.1f}%")

In [ ]:
# ── PCA scatter coloured by cluster ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
palette = sns.color_palette('tab10', OPTIMAL_K)

for k in range(OPTIMAL_K):
    mask = cluster_labels == k
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=[palette[k]], alpha=0.4, s=18, label=f'Cluster {k}')

# Plot centroids
centroids_pca = pca.transform(km_final.cluster_centers_)
ax.scatter(centroids_pca[:,0], centroids_pca[:,1],
           c='black', marker='X', s=200, zorder=5, label='Centroids')

ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)')
ax.set_title(f'K-Means Clusters (k={OPTIMAL_K}) — PCA 2D Projection')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cluster profiling + naming ────────────────────────────────────────────────
profile_cols = ['total_orders','total_sales','total_profit','avg_discount',
                'avg_satisfaction','return_rate','avg_days_ship','avg_age']
profile = cust_features.groupby('cluster')[profile_cols].mean().round(2)

print("Cluster Profiles (mean values):")
print(profile.to_string())

# Heatmap
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(profile.T, annot=True, fmt='.1f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5,
            cbar_kws={'label':'Scaled Mean Value'})
ax.set_title('Cluster Profile Heatmap — Mean Feature Values per Cluster')
plt.tight_layout()
plt.show()

# Business names based on profiles
print("\n📋 Suggested Cluster Names (review profile above to confirm):")
print("  Highest total_sales + low return_rate → 'High-Value Champions'")
print("  High avg_discount + low satisfaction  → 'Bargain Hunters'")
print("  Low frequency + medium sales          → 'Occasional Buyers'")
print("  Low sales + low satisfaction          → 'Dormant Customers'")

---
## Task 4 — Hierarchical Clustering
📘 **Theory:** Hierarchical clustering builds a tree of clusters (a **dendrogram**) by repeatedly merging the two closest clusters.

**Ward's linkage**: Merges clusters to minimise the increase in total within-cluster variance. Produces compact, roughly equal clusters — the best general choice.

**Reading a dendrogram**: The height of a merge shows the distance (dissimilarity) between the two groups being merged. A horizontal cut at a high distance typically reveals the natural number of clusters.

**Adjusted Rand Index (ARI)**: Compares two clusterings. ARI=1 = perfect agreement, ARI=0 = random, ARI<0 = less agreement than random.

In [ ]:
# Use a subsample for speed (hierarchical is O(n²))
SAMPLE_N = min(800, len(X_scaled))
idx_sample = np.random.choice(len(X_scaled), SAMPLE_N, replace=False)
X_hier = X_scaled[idx_sample]
labels_km_sample = cluster_labels[idx_sample]

# Compute linkage matrix
Z = linkage(X_hier, method='ward')
print(f"Linkage matrix shape: {Z.shape}")
print(f"(Each row: [idx1, idx2, distance, count_merged])")
print(f"\nLast 5 merges:")
print(pd.DataFrame(Z[-5:], columns=['idx1','idx2','distance','count']).round(3).to_string(index=False))

In [ ]:
# ── Plot dendrogram ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(Z, ax=ax, truncate_mode='lastp', p=30, leaf_rotation=90,
           color_threshold=Z[-OPTIMAL_K+1, 2])  # cut at k clusters

# Draw cut line
cut_height = Z[-OPTIMAL_K+1, 2]
ax.axhline(y=cut_height, color='red', linestyle='--', linewidth=2,
           label=f'Cut at height={cut_height:.2f} → {OPTIMAL_K} clusters')

ax.set_title(f'Hierarchical Clustering Dendrogram (Ward linkage)\n'
             f'Red line = cut height for {OPTIMAL_K} clusters')
ax.set_xlabel('Samples (or cluster size in brackets)')
ax.set_ylabel('Ward Distance')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Extract clusters and compare with K-Means ─────────────────────────────────
hier_labels = fcluster(Z, t=OPTIMAL_K, criterion='maxclust') - 1  # 0-indexed

ari = adjusted_rand_score(labels_km_sample, hier_labels)
print(f"Adjusted Rand Index (K-Means vs Hierarchical): {ari:.4f}")
print(f"Interpretation: {ari:.4f} — ", end='')
if   ari > 0.8: print("Very high agreement — both methods find similar structure")
elif ari > 0.5: print("Moderate agreement — broadly similar but some differences")
elif ari > 0.2: print("Low agreement — methods find different structures")
else:           print("Near-random agreement — very different partitions")

print("\nHierarchical cluster sizes:")
unique, counts = np.unique(hier_labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Cluster {u}: {c} customers")

---
## Task 5 — DBSCAN Clustering
📘 **Theory:** DBSCAN (Density-Based Spatial Clustering of Applications with Noise) groups together points that are *densely packed* and marks outliers as noise.

**Key parameters:**
- **eps**: The radius of the neighbourhood around a point
- **min_samples**: Minimum points needed in the neighbourhood to form a cluster core

**Choosing eps with the k-distance graph**: Sort distances to each point's k-th nearest neighbour. The "knee" of this curve is the ideal eps value.

**Noise points (label = −1)**: Points that don't belong to any cluster. These are genuine outlier customers — useful for fraud detection or anomaly investigation.

In [ ]:
# ── k-distance graph to choose eps ───────────────────────────────────────────
from sklearn.neighbors import NearestNeighbors

# Work in 2D PCA space for speed
X_2d = X_pca  # already computed above
k = 4

nbrs = NearestNeighbors(n_neighbors=k).fit(X_2d)
distances, _ = nbrs.kneighbors(X_2d)
k_distances = np.sort(distances[:, k-1])[::-1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(k_distances, color='steelblue', linewidth=1.5)
ax.set_xlabel('Points (sorted by distance)')
ax.set_ylabel(f'Distance to {k}th Nearest Neighbour')
ax.set_title('k-Distance Graph — Choose eps at the "knee"')

# Find knee point approximately
knee_idx = np.argmax(np.diff(np.diff(k_distances)))
eps_suggested = k_distances[knee_idx]
ax.axhline(eps_suggested, color='red', linestyle='--',
           label=f'Suggested eps ≈ {eps_suggested:.2f}')
ax.legend()
plt.tight_layout()
plt.show()
print(f"Suggested eps: {eps_suggested:.3f}")

In [ ]:
# ── Fit DBSCAN ────────────────────────────────────────────────────────────────
dbscan = DBSCAN(eps=eps_suggested, min_samples=4)
db_labels = dbscan.fit_predict(X_2d)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise    = (db_labels == -1).sum()
noise_pct  = n_noise / len(db_labels) * 100

print(f"DBSCAN results:")
print(f"  Clusters found: {n_clusters}")
print(f"  Noise points:   {n_noise} ({noise_pct:.1f}%)")
print(f"  Label distribution: {dict(zip(*np.unique(db_labels, return_counts=True)))}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# DBSCAN plot
palette_db = sns.color_palette('tab10', n_clusters)
for lbl in sorted(set(db_labels)):
    mask = db_labels == lbl
    color = 'lightgrey' if lbl == -1 else palette_db[lbl]
    marker = 'x' if lbl == -1 else 'o'
    label = f'Noise ({n_noise})' if lbl == -1 else f'Cluster {lbl}'
    axes[0].scatter(X_2d[mask,0], X_2d[mask,1], c=[color], marker=marker,
                    alpha=0.4 if lbl==-1 else 0.6, s=15, label=label)
axes[0].set_title(f'DBSCAN — {n_clusters} clusters + {n_noise} noise points')
axes[0].legend(fontsize=8)

# K-Means for comparison
for k in range(OPTIMAL_K):
    mask = cluster_labels == k
    axes[1].scatter(X_pca[mask,0], X_pca[mask,1], alpha=0.4, s=15,
                    c=[sns.color_palette('tab10',OPTIMAL_K)[k]], label=f'K-Means {k}')
axes[1].set_title(f'K-Means (k={OPTIMAL_K}) — for comparison')
axes[1].legend(fontsize=8)

plt.suptitle('DBSCAN vs K-Means — Same PCA 2D Space', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
# 📈 Assignment 5 — Time Series Forecasting
**Marks: 100 | Tasks: 7**

### 📘 What is Time Series Forecasting?
A time series is a sequence of values ordered in time. Forecasting predicts future values based on historical patterns.

**Three components of a time series:**
- **Trend**: Long-term direction (increasing/decreasing sales over years)
- **Seasonality**: Regular, periodic patterns (Q4 holiday spike every year)
- **Residual**: Random noise that cannot be explained by trend or seasonality

**Three models we'll build:**
| Model | Type | Captures |
|---|---|---|
| **ARIMA** | Statistical | Trend + autocorrelation |
| **SARIMA** | Statistical | Trend + autocorrelation + seasonality |
| **Prophet** | Additive | Trend + multiple seasonalities + holidays |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi':110})

df = pd.read_csv('retail_sales_clean.csv', parse_dates=['order_date'])
df_ts = df.set_index('order_date').sort_index()

# Create monthly time series
monthly = df_ts['sales_amount'].resample('ME').sum()
print(f"Monthly time series: {len(monthly)} observations")
print(f"Period: {monthly.index[0].strftime('%b %Y')} to {monthly.index[-1].strftime('%b %Y')}")
print(f"\nFirst 6 months:")
print(monthly.head(6).apply(lambda x: f'₹{x:,.0f}').to_string())

---
## Task 1 — Time Series Preparation
📘 **Theory:** Raw transaction data must be *resampled* into a regular time index before modelling. Missing time periods (months with no data) must be interpolated — models require a complete, regularly-spaced series.

In [ ]:
# ── Check for missing months ──────────────────────────────────────────────────
full_index = pd.date_range(monthly.index.min(), monthly.index.max(), freq='ME')
missing_months = full_index.difference(monthly.index)
print(f"Missing months in the time series: {len(missing_months)}")
if len(missing_months) > 0:
    monthly = monthly.reindex(full_index).interpolate(method='time')
    print("Filled with time-based interpolation.")

# ── Plot the time series ──────────────────────────────────────────────────────
rolling_3 = monthly.rolling(window=3, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly, color='steelblue', alpha=0.7, linewidth=1.5, label='Monthly Sales')
ax.plot(rolling_3, color='red', linewidth=2.5, label='3-Month Rolling Mean')
ax.fill_between(monthly.index, monthly, alpha=0.1, color='steelblue')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))
ax.set_title('Monthly Sales Amount — 2020 to 2024')
ax.set_xlabel('Month'); ax.set_ylabel('Sales (₹M)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Task 2 — Time Series Decomposition
📘 **Theory:** Decomposition separates the time series into its three components so we can study each independently.

**STL (Seasonal-Trend decomposition using LOESS)** is more robust than classical decomposition because:
- It handles any seasonality period
- It is robust to outliers (uses LOESS smoothing)
- Trend and seasonal components can change over time

**Seasonality Strength:**
$$F_s = \max\left(0, 1 - \frac{\text{Var}(R)}{\text{Var}(S+R)}\right)$$

Values near 1.0 = strong seasonality. Values near 0 = weak/no seasonality.

In [ ]:
# ── STL Decomposition ─────────────────────────────────────────────────────────
stl = STL(monthly, period=12, seasonal=13)
stl_result = stl.fit()

fig = stl_result.plot()
fig.suptitle('STL Decomposition — Observed | Trend | Seasonal | Residual', fontsize=12, y=1.01)
fig.set_size_inches(14, 9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Seasonality and trend strength ───────────────────────────────────────────
T = stl_result.trend
S = stl_result.seasonal
R = stl_result.resid

Fs = max(0, 1 - np.var(R) / np.var(S + R))
Ft = max(0, 1 - np.var(R) / np.var(T + R))

print(f"Trend Strength     : {Ft:.4f}  ({'Strong' if Ft>0.6 else 'Moderate' if Ft>0.3 else 'Weak'})")
print(f"Seasonality Strength: {Fs:.4f}  ({'Strong' if Fs>0.6 else 'Moderate' if Fs>0.3 else 'Weak'})")

# ── Which months are highest and lowest seasonal factor? ─────────────────────
monthly_seasonal = pd.Series(S.values, index=monthly.index)
monthly_avg_seasonal = monthly_seasonal.groupby(monthly_seasonal.index.month).mean()
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
monthly_avg_seasonal.index = [month_names[m] for m in monthly_avg_seasonal.index]

fig, ax = plt.subplots(figsize=(11, 4))
colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in monthly_avg_seasonal]
monthly_avg_seasonal.plot(kind='bar', ax=ax, color=colors)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Average Seasonal Factor by Month\n(positive = above-trend, negative = below-trend)')
ax.set_xlabel('Month'); ax.set_ylabel('Seasonal Factor (INR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x:,.0f}'))
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

print("\nTop 3 seasonal months:", monthly_avg_seasonal.nlargest(3).index.tolist())
print("Lowest 3 seasonal months:", monthly_avg_seasonal.nsmallest(3).index.tolist())

---
## Task 3 — Stationarity Testing
📘 **Theory:** ARIMA models require the time series to be **stationary** — meaning its statistical properties (mean, variance) do not change over time.

**ADF (Augmented Dickey-Fuller) test:**
- H₀: Series has a unit root (non-stationary)
- Reject H₀ (p < 0.05) → series IS stationary

**KPSS test (opposite logic):**
- H₀: Series IS stationary
- Reject H₀ (p < 0.05) → series is NOT stationary

**Differencing**: Subtracting the previous value. `d=1` (first-order difference) removes linear trends. `D=1` (seasonal difference at lag 12) removes annual seasonality.

In [ ]:
def stationarity_report(series, name):
    """Run ADF and KPSS tests."""
    # ADF
    adf_result = adfuller(series.dropna(), autolag='AIC')
    adf_stat, adf_p = adf_result[0], adf_result[1]

    # KPSS
    kpss_stat, kpss_p, _, _ = kpss(series.dropna(), regression='c', nlags='auto')

    print(f"\n{'='*55}")
    print(f"  Stationarity Test: {name}")
    print(f"{'='*55}")
    print(f"  ADF  test:  stat={adf_stat:.4f}  p={adf_p:.4f}  "
          f"→ {'Stationary ✅' if adf_p<0.05 else 'NON-Stationary ❌'}")
    print(f"  KPSS test:  stat={kpss_stat:.4f}  p={kpss_p:.4f}  "
          f"→ {'NON-Stationary ❌' if kpss_p<0.05 else 'Stationary ✅'}")

    if adf_p < 0.05 and kpss_p >= 0.05:
        print("  Conclusion: STATIONARY (both tests agree) ✅")
        return True
    elif adf_p >= 0.05 and kpss_p < 0.05:
        print("  Conclusion: NON-STATIONARY (both tests agree) ❌  → Apply differencing")
        return False
    else:
        print("  Conclusion: Mixed signals — proceed with differencing as precaution")
        return False

is_stat = stationarity_report(monthly, 'Original Monthly Series')

In [ ]:
# ── Apply differencing if needed ─────────────────────────────────────────────
monthly_d = monthly.diff().dropna()
print("After first-order differencing:")
is_stat_d = stationarity_report(monthly_d, 'First-Differenced Series')

# Seasonal differencing if still non-stationary
monthly_sd = monthly_d.diff(12).dropna()
print("\nAfter seasonal differencing (lag=12):")
is_stat_sd = stationarity_report(monthly_sd, 'Seasonally-Differenced Series')

# Determine d and D
d = 0 if is_stat else 1
D = 0 if is_stat_d else 1
print(f"\nConclusion: d={d} (non-seasonal differencing order)")
print(f"           D={D} (seasonal differencing order)")

---
## Task 4 — ARIMA Modelling
📘 **Theory — ARIMA(p, d, q):**
- **p** (AR order): Number of lag observations — found from PACF. Use lags where PACF crosses outside the blue confidence band.
- **d** (Integration order): Determined from stationarity tests (Task 3).
- **q** (MA order): Moving average window — found from ACF of differenced series.

**ACF (Autocorrelation Function)**: Correlation of the series with itself at various lags. Significant spikes suggest MA terms.
**PACF (Partial ACF)**: Correlation at each lag after removing shorter-lag effects. Significant spikes suggest AR terms.

**Residual diagnostics:** Good model residuals should look like white noise — no patterns, approximately normal, no autocorrelation.

In [ ]:
# ── ACF and PACF plots ────────────────────────────────────────────────────────
work_series = monthly_d  # use differenced series

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(work_series, lags=24, ax=axes[0], alpha=0.05, zero=False)
axes[0].set_title('ACF — Guides MA order (q)\nSignificant spike at lag k → include MA(k)')

plot_pacf(work_series, lags=24, ax=axes[1], alpha=0.05, zero=False, method='ywm')
axes[1].set_title('PACF — Guides AR order (p)\nSignificant spike at lag k → include AR(k)')

plt.suptitle('Autocorrelation Functions — Differenced Monthly Sales', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()
print("Reading: Blue band = 95% confidence interval. Bars outside the band are significant.")
print("PACF cuts off after lag p → AR(p). ACF cuts off after lag q → MA(q).")

In [ ]:
# ── Fit ARIMA ─────────────────────────────────────────────────────────────────
# Try auto_arima first for guidance
try:
    from pmdarima import auto_arima
    auto_model = auto_arima(monthly, seasonal=False, stepwise=True,
                             information_criterion='aic', error_action='ignore',
                             suppress_warnings=True)
    p_auto, d_auto, q_auto = auto_model.order
    print(f"auto_arima suggestion: ARIMA({p_auto},{d_auto},{q_auto})  AIC={auto_model.aic():.2f}")
except ImportError:
    p_auto, d_auto, q_auto = 1, 1, 1
    print(f"pmdarima not available — using ARIMA({p_auto},{d_auto},{q_auto})")

# Fit with statsmodels
arima_model = ARIMA(monthly, order=(p_auto, d_auto, q_auto))
arima_result = arima_model.fit()
print(f"\nARIMA({p_auto},{d_auto},{q_auto}) Results:")
print(f"  AIC = {arima_result.aic:.2f}")
print(f"  BIC = {arima_result.bic:.2f}")
print(f"  Log-Likelihood = {arima_result.llf:.2f}")
print("\nCoefficients (p-values):")
summary = arima_result.summary().tables[1]
print(summary)

In [ ]:
# ── ARIMA residual diagnostics ────────────────────────────────────────────────
from scipy import stats as sp

resid = arima_result.resid.dropna()

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Residuals over time
axes[0,0].plot(resid, color='steelblue', linewidth=1)
axes[0,0].axhline(0, color='red', linestyle='--')
axes[0,0].set_title('Residuals Over Time\n(should look like white noise)')

# Residual histogram
sns.histplot(resid, kde=True, bins=25, ax=axes[0,1], color='darkorange')
axes[0,1].set_title('Residual Distribution\n(should be bell-shaped / normal)')

# ACF of residuals
plot_acf(resid, lags=20, ax=axes[1,0], alpha=0.05, zero=False)
axes[1,0].set_title('ACF of Residuals\n(no significant spikes = white noise ✅)')

# Q-Q plot
sp.probplot(resid, dist='norm', plot=axes[1,1])
axes[1,1].set_title('Q-Q Plot of Residuals\n(points on line = normal ✅)')

plt.suptitle('ARIMA Residual Diagnostics', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Ljung-Box test
from statsmodels.stats.diagnostic import acorr_ljungbox
lb_test = acorr_ljungbox(resid, lags=[10,20], return_df=True)
print("Ljung-Box test (H₀: residuals are white noise):")
print(lb_test.to_string())
print("p > 0.05 → fail to reject H₀ → residuals are white noise ✅")

In [ ]:
# ── ARIMA forecast (12 months) ────────────────────────────────────────────────
n_forecast = 12
forecast_arima = arima_result.get_forecast(steps=n_forecast)
fc_mean   = forecast_arima.predicted_mean
fc_ci     = forecast_arima.conf_int(alpha=0.05)
fc_dates  = pd.date_range(monthly.index[-1] + pd.DateOffset(months=1),
                           periods=n_forecast, freq='ME')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly, color='steelblue', linewidth=2, label='Historical')
ax.plot(fc_dates, fc_mean, color='red', linewidth=2, linestyle='--', label='ARIMA Forecast')
ax.fill_between(fc_dates, fc_ci.iloc[:,0], fc_ci.iloc[:,1],
                color='red', alpha=0.15, label='95% CI')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))
ax.set_title(f'ARIMA({p_auto},{d_auto},{q_auto}) — 12-Month Forecast')
ax.legend()
ax.axvline(monthly.index[-1], color='grey', linestyle=':', linewidth=1.5)
plt.tight_layout()
plt.show()

---
## Task 5 — SARIMA Modelling
📘 **Theory — SARIMA(p,d,q)(P,D,Q,s):**

SARIMA extends ARIMA by adding *seasonal* AR, differencing, and MA terms. For monthly data: **s=12**.

- **(P)**: Seasonal AR order — uses lags at multiples of s (e.g., lag 12, 24)
- **(D)**: Seasonal differencing — removes annual seasonality
- **(Q)**: Seasonal MA order

SARIMA directly models the seasonal structure, making it generally better than ARIMA for monthly retail data with strong annual seasonality.

In [ ]:
# ── Fit SARIMA ────────────────────────────────────────────────────────────────
try:
    sarima_auto = auto_arima(monthly, seasonal=True, m=12, stepwise=True,
                              information_criterion='aic', error_action='ignore',
                              suppress_warnings=True)
    sp_order = sarima_auto.seasonal_order
    ns_order = sarima_auto.order
    print(f"auto_arima SARIMA suggestion: {ns_order} × {sp_order}")
except:
    ns_order, sp_order = (1,1,1), (1,1,1,12)
    print(f"Using default SARIMA: {ns_order} × {sp_order}")

sarima_result = SARIMAX(monthly, order=ns_order,
                         seasonal_order=sp_order).fit(disp=False)

print(f"\nSARIMA Results:")
print(f"  AIC = {sarima_result.aic:.2f}  (ARIMA AIC = {arima_result.aic:.2f})")
print(f"  BIC = {sarima_result.bic:.2f}  (ARIMA BIC = {arima_result.bic:.2f})")
print(f"  SARIMA AIC {'lower (better) ✅' if sarima_result.aic < arima_result.aic else 'higher than ARIMA'}")

In [ ]:
# ── SARIMA 2025 forecast ──────────────────────────────────────────────────────
sarima_forecast = sarima_result.get_forecast(steps=12)
fc_sarima_mean = sarima_forecast.predicted_mean
fc_sarima_ci   = sarima_forecast.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly, color='steelblue', linewidth=2, label='Historical')
ax.plot(fc_dates, fc_sarima_mean, color='#27ae60', linewidth=2.5,
        linestyle='--', label='SARIMA Forecast')
ax.fill_between(fc_dates, fc_sarima_ci.iloc[:,0], fc_sarima_ci.iloc[:,1],
                color='#27ae60', alpha=0.15, label='95% CI')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))
ax.set_title(f'SARIMA{ns_order}×{sp_order} — 12-Month Forecast (2025)')
ax.legend()
ax.axvline(monthly.index[-1], color='grey', linestyle=':', linewidth=1.5)
plt.tight_layout()
plt.show()

---
## Task 6 — Facebook Prophet Forecasting
📘 **Theory:** Prophet decomposes the time series as:
$$y(t) = \text{trend}(t) + \text{seasonality}(t) + \text{holidays}(t) + \epsilon$$

**Advantages over ARIMA:**
- Handles missing data and outliers robustly
- Automatically detects changepoints (trend changes)
- Easy to add custom seasonalities and external regressors
- Does not require stationarity preprocessing

**Changepoints**: Dates where the trend significantly changed direction. `changepoint_prior_scale` controls flexibility — higher values = more flexible trend.

In [ ]:
try:
    from prophet import Prophet

    # Prepare Prophet-format DataFrame
    df_prophet = pd.DataFrame({'ds': monthly.index, 'y': monthly.values})

    # Fit default model
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                daily_seasonality=False, changepoint_prior_scale=0.05)
    m.add_seasonality(name='quarterly', period=91.25, fourier_order=5)
    m.fit(df_prophet)

    # Forecast 12 months
    future = m.make_future_dataframe(periods=12, freq='ME')
    forecast_prophet = m.predict(future)

    # Plot
    fig = m.plot(forecast_prophet, figsize=(14, 5))
    plt.title('Facebook Prophet — Historical + 12-Month Forecast', fontsize=12)
    plt.gca().yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))
    plt.tight_layout()
    plt.show()

    # Components
    fig_comp = m.plot_components(forecast_prophet)
    plt.suptitle('Prophet Components: Trend + Seasonality', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

    prophet_available = True
    print("Prophet model fitted successfully ✅")

except ImportError:
    print("Prophet not installed. Run: pip install prophet")
    print("Skipping Prophet section — will use SARIMA as the 'best model' below.")
    prophet_available = False

---
## Task 7 — Model Comparison & Final 2025 Forecast
📘 **Theory — Evaluation Metrics:**

| Metric | Formula | Sensitive to |
|---|---|---|
| **MAE** | mean(\|y − ŷ\|) | Average absolute error |
| **RMSE** | √mean((y−ŷ)²) | Large errors (penalises outliers more) |
| **MAPE** | mean(\|y−ŷ\|/\|y\|)×100 | Relative error in % |
| **SMAPE** | mean(2\|y−ŷ\|/(\|y\|+\|ŷ\|))×100 | Symmetric MAPE (handles near-zero actuals) |

**Train/Test split:** Use the last 6 months as a held-out test set. Train all models on the same training window, then forecast into the test period.

In [ ]:
# ── Train / Test split ────────────────────────────────────────────────────────
TRAIN_END = '2024-06-30'
TEST_START = '2024-07-01'

train = monthly[:TRAIN_END]
test  = monthly[TEST_START:]
n_test = len(test)

print(f"Train: {train.index[0].strftime('%b %Y')} to {train.index[-1].strftime('%b %Y')} ({len(train)} months)")
print(f"Test:  {test.index[0].strftime('%b %Y')}  to {test.index[-1].strftime('%b %Y')}  ({n_test} months)")

In [ ]:
def eval_metrics(actual, predicted, model_name):
    """Compute MAE, RMSE, MAPE, SMAPE."""
    mae   = mean_absolute_error(actual, predicted)
    rmse  = np.sqrt(mean_squared_error(actual, predicted))
    mape  = np.mean(np.abs((actual - predicted) / actual)) * 100
    smape = np.mean(2 * np.abs(actual - predicted) / (np.abs(actual) + np.abs(predicted))) * 100
    return {'Model': model_name, 'MAE': mae, 'RMSE': rmse,
            'MAPE %': mape, 'SMAPE %': smape}

results = []

# ── ARIMA on train ────────────────────────────────────────────────────────────
arima_train = ARIMA(train, order=(p_auto,d_auto,q_auto)).fit()
fc_arima_test = arima_train.forecast(steps=n_test)
results.append(eval_metrics(test.values, fc_arima_test.values, 'ARIMA'))

# ── SARIMA on train ───────────────────────────────────────────────────────────
sarima_train = SARIMAX(train, order=ns_order, seasonal_order=sp_order).fit(disp=False)
fc_sarima_test = sarima_train.forecast(steps=n_test)
results.append(eval_metrics(test.values, fc_sarima_test.values, 'SARIMA'))

# ── Prophet on train (if available) ───────────────────────────────────────────
if prophet_available:
    df_p_train = pd.DataFrame({'ds': train.index, 'y': train.values})
    m_train = Prophet(yearly_seasonality=True, changepoint_prior_scale=0.05)
    m_train.add_seasonality(name='quarterly', period=91.25, fourier_order=5)
    m_train.fit(df_p_train)
    future_test = m_train.make_future_dataframe(periods=n_test, freq='ME')
    fc_prophet = m_train.predict(future_test)
    fc_prophet_test = fc_prophet.set_index('ds')['yhat'].loc[test.index]
    results.append(eval_metrics(test.values, fc_prophet_test.values, 'Prophet'))

comparison = pd.DataFrame(results).round(2)
print("\n=== MODEL COMPARISON ===")
print(comparison.to_string(index=False))
best_model = comparison.loc[comparison['MAPE %'].idxmin(), 'Model']
print(f"\nBest model by MAPE: {best_model} ✅")

In [ ]:
# ── Plot all forecasts vs actual test ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

# Training data
ax.plot(train[-18:], color='steelblue', linewidth=2, label='Training (last 18 months)')

# Actual test
ax.plot(test, color='black', linewidth=2.5, marker='o', markersize=6, label='Actual Test')

# ARIMA forecast
ax.plot(test.index, fc_arima_test.values, color='#e74c3c',
        linestyle='--', linewidth=2, marker='s', markersize=5, label='ARIMA Forecast')

# SARIMA forecast
ax.plot(test.index, fc_sarima_test.values, color='#27ae60',
        linestyle='--', linewidth=2, marker='^', markersize=5, label='SARIMA Forecast')

if prophet_available:
    ax.plot(test.index, fc_prophet_test.values, color='#9b59b6',
            linestyle='--', linewidth=2, marker='D', markersize=5, label='Prophet Forecast')

ax.axvline(train.index[-1], color='grey', linestyle=':', linewidth=2, label='Train/Test split')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))
ax.set_title('All Models vs Actual — Test Period Comparison')
ax.set_xlabel('Month'); ax.set_ylabel('Sales (₹M)')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()
print(comparison.to_string(index=False))

In [ ]:
# ── Final 2025 Forecast using best model ─────────────────────────────────────
print(f"Generating 2025 forecast using {best_model}...")

if best_model == 'SARIMA':
    final_result = SARIMAX(monthly, order=ns_order, seasonal_order=sp_order).fit(disp=False)
    fc_2025 = final_result.get_forecast(steps=12)
    fc_mean   = fc_2025.predicted_mean
    fc_ci     = fc_2025.conf_int(alpha=0.05)
elif best_model == 'Prophet' and prophet_available:
    m_full = Prophet(yearly_seasonality=True, changepoint_prior_scale=0.05)
    m_full.add_seasonality(name='quarterly', period=91.25, fourier_order=5)
    m_full.fit(pd.DataFrame({'ds': monthly.index, 'y': monthly.values}))
    future_2025 = m_full.make_future_dataframe(periods=12, freq='ME')
    fc_df = m_full.predict(future_2025)[-12:]
    fc_mean = fc_df.set_index('ds')['yhat']
    fc_ci   = fc_df.set_index('ds')[['yhat_lower','yhat_upper']]
else:
    final_result = ARIMA(monthly, order=(p_auto,d_auto,q_auto)).fit()
    fc_2025   = final_result.get_forecast(steps=12)
    fc_mean   = fc_2025.predicted_mean
    fc_ci     = fc_2025.conf_int(alpha=0.05)

# ── 2025 forecast table ───────────────────────────────────────────────────────
import calendar
mom_change = fc_mean.pct_change() * 100
forecast_table = pd.DataFrame({
    'Month':      [f"{calendar.month_abbr[d.month]} {d.year}" for d in fc_mean.index],
    'Forecast ₹M': (fc_mean.values / 1e6).round(2),
    'Lower 95% ₹M': (fc_ci.iloc[:,0].values / 1e6).round(2),
    'Upper 95% ₹M': (fc_ci.iloc[:,1].values / 1e6).round(2),
    'MoM Change %': mom_change.values.round(1),
})
print(f"\n2025 Monthly Sales Forecast ({best_model}):")
print(forecast_table.to_string(index=False))
print(f"\nProjected 2025 Total: ₹{fc_mean.sum()/1e6:.1f}M")

In [ ]:
# ── Final 2025 forecast chart ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 6))

# Historical
ax.plot(monthly, color='steelblue', linewidth=2, alpha=0.8, label='Historical Sales')
ax.fill_between(monthly.index, monthly, alpha=0.1, color='steelblue')

# Forecast
ax.plot(fc_mean.index, fc_mean.values, color='#e74c3c', linewidth=2.5,
        linestyle='--', marker='o', markersize=6, label=f'2025 Forecast ({best_model})')
ax.fill_between(fc_mean.index, fc_ci.iloc[:,0], fc_ci.iloc[:,1],
                color='#e74c3c', alpha=0.15, label='95% Confidence Interval')

# Vertical line at forecast start
ax.axvline(monthly.index[-1], color='grey', linestyle=':', linewidth=2, label='Forecast Start')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))
ax.set_title(f'📈 2025 Monthly Sales Forecast — {best_model} Model\n'
             f'Projected Annual Total: ₹{fc_mean.sum()/1e6:.1f}M',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Sales (₹M)')
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ Assignment 5 Complete!")
print(f"   Model: {best_model}")
print(f"   2025 Peak Month: {forecast_table.loc[forecast_table['Forecast ₹M'].idxmax(), 'Month']}")
print(f"   2025 Total Projection: ₹{fc_mean.sum()/1e6:.1f}M")

---
## 🎉 All 5 Assignments Complete!

| Assignment | Status |
|---|---|
| 1. Data Cleaning | ✅ Done |
| 2. EDA / Analysis | ✅ Done |
| 3. Statistical Analysis | ✅ Done |
| 4. Segmentation & Clustering | ✅ Done |
| 5. Time Series Forecasting | ✅ Done |

### Key Outputs
- `retail_sales_clean.csv` — cleaned, validated dataset
- RFM customer segments with business labels
- K-Means cluster profile heatmap with named segments
- Final 2025 monthly sales forecast table and chart

### Next Steps
- Re-run with fresh data each month to update the forecast
- Use the RFM segments to design targeted email campaigns
- Monitor the MAPE on live data to detect model drift